# Barbados R.O.A.D. — **MEGA notebook**: shared + class-wise + sub-class-wise models, validated every 100 steps, pooled by MBR

One run trains every system in `CFG.SYSTEMS`. Each system's N-best lists are pooled, and the pool is selected by **minimum Bayes risk under the official metric**.

| System (default) | Scheme | LoRA adapters | Prompt |
|---|---|---|---|
| `8b_shared` | one model for all lines | 1 | original prompt (measured 0.9095 / MBR 0.9112) |
| `8b_class` | one fresh model per class | A, B | one prompt per era |
| `8b_subclass` | one fresh model per sub-class | A1, A2, A3, B | one prompt per sub-class |
| `7b_shared` | one model for all lines | 1 | original prompt (measured 0.9055 / MBR 0.9073) |

Optional systems are in the CFG comments: a B specialist continued from `8b_shared`, and a 32B shared model.
A system can also cover only some classes (`keys=[...]`). It then votes only on the lines of those classes.

### Validation every `EVAL_EVERY = 100` steps (inner slice of TRAIN, never the Fold-0 val)
Each adapter is checked at every 100 steps and at its last step. Each check prints:
- its training loss;
- the teacher-forced loss on held-out inner lines of its class;
- the **official score** of greedy transcriptions of up to 160 of them.

The best step by `CKPT_METRIC` is kept in memory and restored at the end, which answers "more epochs got worse". The curves are saved to `eval_curves.csv`.
`MONITOR_VAL_N > 0` also decodes some Fold-0 val lines at each check. This is **printed only**: selecting on it would leak the benchmark.

### Why pool, and why rescore (measured on the EXP_002 outputs, `work/selection_experiments.py`)
**The pooled 7B+8B MBR scores 0.9175.** The best candidate already in the pool would score **0.9497** (the oracle). Can a better selection rule close that gap, using only what the pool already contains?

| Selection rule, honest 5-fold CV over lines | Δ vs pooled MBR |
|---|---|
| Posterior temperature | −0.0005 |
| Per-class system weights | 0.0000 |
| Character 6-gram LM prior (train labels) | 0.0000 (the LM only hurts) |
| Out-of-vocabulary penalty | −0.0002 |
| Median-string search (no knob) | +0.0003, P = 0.78 |
| Gradient-boosted reranker (earlier) | −0.0028 |

Selection on the existing evidence is saturated. The candidate the oracle wants is **in only one model's 5-best list in about 85% of the lines**, so the other model gives it zero weight.
Two things add real evidence:
1. **More, diverse systems.** Class and sub-class models make different errors. Going from one model to two added +0.0064.
2. **Cross-rescoring (`RESCORE=True`).** Every system computes the exact log-probability of every pooled candidate, so every candidate gets a vote from every model. MBR (`rescore_mbr`) and product-of-experts MAP (`rescore_map`) then run over the rescored pool.

Every method is one trial row on the 819 val lines, with a paired bootstrap against `pool_mbr_quality` (the measured champion recipe). The best one writes `submission.csv`, and its settings go to `mega_champion.json` for the `RUN_MODE="full"` refit.

### Runtime (RTX PRO 6000, from EXP_002 timings)
- About 20 min of training per 8B system and 40 min per 7B system.
- About 1 min per validation check.
- About 20 min of decoding per system (val + test + unlisted).
- About 10 min of rescoring per system.

Default 4 systems ≈ 4–5 h. `TIME_BUDGET_H` skips whole base models, and the rescoring pass, when time runs short. Everything already produced stays on disk.

In [ ]:
# =========================================================
# Cell 1 — OFFLINE install from the wheels dataset
# =========================================================
import os, sys, subprocess, glob

os.environ["HF_HUB_OFFLINE"]         = "1"
os.environ["TRANSFORMERS_OFFLINE"]   = "1"
os.environ["HF_DATASETS_OFFLINE"]    = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import glob as _glob
import os as _os
WHEELS = "/kaggle/input/datasets/haniagamal/teleco-wheels/wheels"
if not _glob.glob(_os.path.join(WHEELS, "*.whl")):
    _hits = _glob.glob("/kaggle/input/**/*.whl", recursive=True)
    if _hits:
        WHEELS = _os.path.dirname(sorted(_hits, key=len)[0])
print("wheels dir:", WHEELS, "  found:",
      len(_glob.glob(_os.path.join(WHEELS, "*.whl"))), "wheels")

In [ ]:
%%capture
!pip uninstall -y torchao
!pip install --no-index --find-links {WHEELS} \
    transformers datasets peft trl jiwer timm librosa soundfile \
    accelerate bitsandbytes huggingface_hub torchao==0.16
print("Dependencies installed (offline).")

In [ ]:
# =========================================================
# Cell 2 — imports
# =========================================================
import ast, glob, json, shutil, gc, time, math, re, copy, collections
import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
import numpy as np
import pandas as pd
from PIL import Image
from glob import glob as _glob
from tqdm.auto import tqdm
from sklearn.model_selection import KFold

import transformers
from datasets import Dataset
from transformers import AutoProcessor, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel

try:
    from rapidfuzz.distance import Levenshtein as _RFLev      # ships with jiwer>=3
    def _lev(a, b):
        return _RFLev.distance(a, b)
except Exception:                                              # pure-python fallback
    def _lev(a, b):
        n, m = len(a), len(b)
        if n == 0: return m
        if m == 0: return n
        prev = list(range(m + 1))
        for i in range(1, n + 1):
            cur = [i] + [0] * m; ai = a[i - 1]
            for j in range(1, m + 1):
                cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ai != b[j - 1]))
            prev = cur
        return prev[m]

def _auto_vlm_class():
    for name in ("AutoModelForImageTextToText", "AutoModelForVision2Seq"):
        if hasattr(transformers, name):
            return getattr(transformers, name)
    raise ImportError("no AutoModelForImageTextToText / AutoModelForVision2Seq")
AutoVLM = _auto_vlm_class()

print("transformers", transformers.__version__, "| VLM class:", AutoVLM.__name__)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| bf16", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
SESSION_T0 = time.time()

In [ ]:
# =========================================================
# Cell 3 — CFG for the MEGA run
# =========================================================
TRIAL_ID          = "MEGA_001"
TRIAL_DESCRIPTION = ("Shared + class-wise (A/B) + sub-class-wise (A1/A2/A3/B) Qwen3-VL-8B systems and a shared Qwen2.5-VL-7B; "
                     "inner validation every 100 steps with best-step restore; beam-5 N-best pooled; cross-system rescoring; "
                     "MBR ensembles on all 819 Fold-0 val lines")

SHARED_PROMPT = ("This is not modern English, Transcribe the handwriting exactly. Keep the wrong spelling, abbreviations, "
                 "and marks (^, ff, unusual letters).Do not modernize or correct anything.")      # measured prompt (EXP_002)
COMMON_PROMPT = ("Transcribe the handwritten line exactly as written. This is not modern English: keep the original "
                 "spelling, capital letters, abbreviations and marks (^ for superscript letters, : for suspensions, "
                 "&, ff). Do not modernize, expand or correct anything. If parts of neighbouring lines are visible, "
                 "transcribe only the central line.")
_A = (" This line comes from a Barbados legal record of the 1630s-1660s. Common conventions in these documents: "
      "& for 'and', ye and yt for 'the' and 'that', sd for 'said', and spellings such as heires, assignes, publique.")
_B = (" This line comes from a Barbados legal record of the 1670s-1710s, scanned at high resolution. Common "
      "conventions in these documents: superscript abbreviations written with ^ (W^m, y^e, M^r, Adm^rs, Ex^rs), "
      "colon suspensions (Tho:, Exec:^rs, S:^d), ff at the start of words, and spellings such as heyres and saide.")

class CFG:
    WHEELS_DIR = WHEELS
    # ---- base models (tag -> path) ----
    BASE_MODELS = {
        "8b":  "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/8b-instruct/1",
        "7b":  "/kaggle/input/models/qwen-lm/qwen2.5-vl/transformers/7b-instruct/2",
        "32b": "/kaggle/input/models/qwen-lm/qwen-3-vl/transformers/32b-instruct/1",
    }
    # ---- SYSTEMS: each one is trained (validated every EVAL_EVERY steps), decoded (N-best) and pooled ----
    #   scheme "shared"  : ONE LoRA on all lines, prompt SHARED_PROMPT
    #   scheme "class"   : one fresh LoRA per class A / B, each trained on its class only, prompt CLASS_PROMPTS[class]
    #   scheme "subclass": one fresh LoRA per sub-class A1 / A2 / A3 / B
    #   optional: keys=[...]           only these classes (the system votes only on their lines)
    #             init="from:<name>"   start every adapter from a SHARED system of the same base model (trained earlier)
    #             replay=0.15          share of other-class lines mixed into each class adapter
    #             hp={...}             LEARNING_RATE, NUM_TRAIN_EPOCHS, MAX_STEPS, R, LORA_ALPHA, AUG_COPIES, ...
    #             class_hp={"A2": {"NUM_TRAIN_EPOCHS": 2}}
    # Systems of the same base model share one loaded base; the order below is the training order.
    SYSTEMS = [
        dict(name="8b_shared",   model="8b", scheme="shared"),
        dict(name="8b_class",    model="8b", scheme="class"),
        dict(name="8b_subclass", model="8b", scheme="subclass"),
        dict(name="7b_shared",   model="7b", scheme="shared"),
        # dict(name="8b_B_cont", model="8b", scheme="class", keys=["B"], init="from:8b_shared", replay=0.15,
        #      hp={"LEARNING_RATE": 1e-4, "LR_SCHEDULER": "constant_with_warmup"}),
        # dict(name="32b_shared", model="32b", scheme="shared"),
    ]
    BASELINE_SYSTEM = "8b_shared"     # every other system / ensemble is compared with this system's top-1
    GRAD_CKPT_MODELS = ("32b",)
    DATA_DIR    = "/kaggle/input/datasets/haniagamal/road-barbados"
    TRAIN_CSV   = None
    TEST_CSV    = None
    IMAGES_DIR  = None
    AUTODISCOVER = True

    # ---- outputs ----
    OUTPUT_DIR     = "/kaggle/working/qwen-vl-ocr"
    FINAL_DIR      = "/kaggle/working/qwen-vl-ocr/final"     # adapters -> FINAL_DIR/<model>/fold0/<adapter>/
    WORK_DIR       = "/kaggle/working"
    SUBMISSION_CSV = "/kaggle/working/submission.csv"
    KEEP_ADAPTERS  = True     # always True in practice: the rescoring pass reloads them
    ADAPTERS_DIR   = None     # DO_TRAIN=False: folder holding <model>/fold0/<adapter>/ ; None -> FINAL_DIR, then /kaggle/input
    PREV_RESULTS_DIRS = []

    # ---- protocol ----
    RUN_MODE        = "fold0"        # "fold0": score ALL 819 val lines | "full": train on all lines, predict test
    FOLD            = 0
    N_FOLDS         = 5
    CKPT_SELECTION  = "inner_loss"   # keeps the inner slice OUT of training (needed by the step validation) — keep
    INNER_CKPT_FRAC = 0.08
    TIME_BUDGET_H   = 11.3

    # ---- validation during training (inner slice of TRAIN) ----
    EVAL_EVERY          = 100        # optimizer steps between validations (the last step is always validated)
    EVAL_GEN_N          = 160        # inner lines greedy-decoded per validation (the adapter's own class)
    EVAL_LOSS_N         = 400        # inner lines for the teacher-forced loss
    EVAL_BATCH          = 16
    EVAL_MAX_NEW_TOKENS = 128
    CKPT_METRIC         = "inner_official"   # "inner_official" | "inner_loss" | "last" — best step restored at the end
    MONITOR_VAL_N       = 0          # >0: also decode this many Fold-0 VAL lines per validation — PRINTED ONLY, never used

    # ---- training (defaults; per model: MODEL_OVERRIDES, per system: hp, per class: class_hp) ----
    PER_DEVICE_TRAIN_BATCH_SIZE  = 4
    GRADIENT_ACCUMULATION_STEPS  = 1
    NUM_TRAIN_EPOCHS             = 1
    MAX_STEPS                    = -1
    LEARNING_RATE                = 2e-4
    LR_SCHEDULER                 = "cosine"
    WARMUP_RATIO                 = 0.03
    DATALOADER_NUM_WORKERS       = 4
    MAX_PIXELS                   = 2_000_000
    MIN_PIXELS                   = 256*28*28
    ATTN_IMPL                    = None
    USE_BF16                     = True
    R                            = 32
    LORA_ALPHA                   = 64
    LORA_DROPOUT                 = 0.05
    LORA_VISION                  = True
    SEED                         = 42
    MODEL_OVERRIDES = {"32b": {"LEARNING_RATE": 1e-4}}

    # ---- classes (pixels only) and prompts ----
    CLASS_HEIGHT_SPLIT  = 150
    GROUP_A1_MAX_HEIGHT = 58
    GROUP_A2_MAX_PAPER  = 194
    CLASS_PROMPTS = {
        "A":  COMMON_PROMPT + _A,
        "B":  COMMON_PROMPT + _B,
        "A1": COMMON_PROMPT + _A + " The crop is tight around the line; '&' and ye/yt are very common.",
        "A2": COMMON_PROMPT + _A + " The parchment is dark and the hand is large; in these documents '&' and ye are rare and lines are short.",
        "A3": COMMON_PROMPT + _A + " The crop is loose and often shows parts of the lines above and below; the central line is usually long.",
    }
    SHARED_PROMPT     = SHARED_PROMPT
    SHARED_CLASS_HINT = False        # True -> shared systems get " Scan type: A." / " B." appended (train + inference)
    CLASS_HINT        = True         # internal: every line's class is passed to the prompt builder — keep True
    HINT_LEVEL        = "group"
    CLASS_AUG_COPIES  = None

    # ---- augmentation (realistic profile) ----
    USE_AUGMENT   = True
    AUG_COPIES    = 1
    AUG_PROFILE   = "realistic"
    AUG_RESAMPLE  = True
    AUG_STROKE_MODE = "both"
    AUG_ROTATE  = True;  AUG_ROTATE_P  = 0.5;  AUG_ROTATE_DEG    = 4.0
    AUG_SHEAR   = True;  AUG_SHEAR_P   = 0.4;  AUG_SHEAR_MAX     = 0.15
    AUG_ELASTIC = True;  AUG_ELASTIC_P = 0.4;  AUG_ELASTIC_ALPHA = 6.0;  AUG_ELASTIC_SIGMA = 8.0
    AUG_HJITTER = True;  AUG_HJITTER_P = 0.4;  AUG_HJITTER_FRAC  = 0.12
    AUG_PAD     = True;  AUG_PAD_P     = 0.3;  AUG_PAD_FRAC      = 0.06
    AUG_PHOTO   = True;  AUG_PHOTO_P   = 0.7;  AUG_BRIGHT = 0.20;  AUG_CONTRAST = 0.20;  AUG_GAMMA = 1.4
    AUG_TINT    = True;  AUG_TINT_P    = 0.5;  AUG_TINT_MAX = 0.12;  AUG_DESAT_MAX = 0.40
    AUG_ILLUM   = True;  AUG_ILLUM_P   = 0.4;  AUG_ILLUM_MAX = 0.35
    AUG_STROKE  = True;  AUG_STROKE_P  = 0.35
    AUG_BLEED   = True;  AUG_BLEED_P   = 0.25;  AUG_BLEED_ALPHA = 0.12
    AUG_NOISE   = True;  AUG_NOISE_P   = 0.5;  AUG_NOISE_STD = 0.1
    AUG_JPEG    = True;  AUG_JPEG_P    = 0.4;  AUG_JPEG_QMIN = 40;  AUG_JPEG_QMAX = 80

    CORRUPT_IDS = {
        "79tMUVyfIdy3GzkG",  "F8DYDDp2AvW9Dytw", "JU7lRwk3jKkus24Z", "VmrEALeZiP1Y6nF9", "t0UrASljcgzvBAnO",
    }

    # ---- decoding: beam 5, 5-best kept (EXP_002: best single-model setting with MBR) + loop guard ----
    MAX_NEW_TOKENS  = 256
    BATCH_SIZE      = 8
    DECODE          = dict(num_beams=5, repetition_penalty=1.0, no_repeat_ngram_size=0, n_return=5)
    DECODE_VARIANTS = {"D2_b5nb": DECODE}
    BASELINE_VARIANT = "D2_b5nb"
    LOOP_GUARD      = 3              # collapse a token repeated more than 3x in a row (7B D1 'x x x ...' loop)
    MBR_TEMPERATURE = 1.0            # offline CV: tuning T gave -0.0005 -> keep 1.0

    # ---- cross-system rescoring ----
    RESCORE        = True
    RESCORE_SPLITS = ("val", "test")           # add "unlisted" for rescored pseudo-labels (+ ~30 % time)
    RESCORE_BATCH  = 8
    RESCORE_T      = 1.0

    # ---- ensembles / final choice ----
    ENSEMBLE_METHODS = ("pool_mbr_equal", "pool_mbr_quality", "pool_mbr_quality_median",
                        "rescore_mbr", "rescore_mbr_median", "rescore_map")
    FINAL_METHOD  = None     # RUN_MODE="full": a method name; None -> read mega_champion.json (FINAL_CHAMPION_JSON)
    FINAL_WEIGHTS = None     # RUN_MODE="full": {system: weight}; None -> from mega_champion.json, else equal
    FINAL_CHAMPION_JSON = None   # path of the Fold-0 run's mega_champion.json (attach its output as a dataset)

    # ---- pseudo-labels ----
    PREDICT_UNLISTED = True
    PSEUDO_LABEL_CSV = None
    PSEUDO_KEEP_FRAC = 0.7
    PSEUDO_PER_CLASS = True

    # ---- flow ----
    DO_TRAIN = True
    DO_INFER = True
    PREDICT_TEST = True
    OCR_PROMPT = SHARED_PROMPT
    # used only by the shared helper cells
    SPECIALISTS = {}; SPECIALIST_MODELS = (); CLASS_SAMPLING_T = None; CLASS_ROUTING = False; ROUTING_LEVEL = "class"

CFG.MODEL_PATHS = list(dict.fromkeys(CFG.BASE_MODELS[s["model"]] for s in CFG.SYSTEMS))
TRIAL_CONFIG = {k: v for k, v in vars(CFG).items() if not k.startswith("_") and not callable(v)}
print("TRIAL:", TRIAL_ID, "|", TRIAL_DESCRIPTION)
print("MODE:", CFG.RUN_MODE, "| systems:", [s["name"] for s in CFG.SYSTEMS], "| base models:", len(CFG.MODEL_PATHS))

In [ ]:
# =========================================================
# Cell 4 — resolve & sanity-check paths
# =========================================================
def _first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

def _find_under(root, filename):
    hits = _glob(os.path.join(root, "**", filename), recursive=True)
    return hits[0] if hits else None

def resolve_paths(cfg):
    train_csv = cfg.TRAIN_CSV or os.path.join(cfg.DATA_DIR, "Train.csv")
    test_csv  = cfg.TEST_CSV  or os.path.join(cfg.DATA_DIR, "Test.csv")
    images    = cfg.IMAGES_DIR or _first_existing([
        os.path.join(cfg.DATA_DIR, "images", "images"),
        os.path.join(cfg.DATA_DIR, "images"),
    ]) or os.path.join(cfg.DATA_DIR, "images", "images")
    if cfg.AUTODISCOVER:
        if not os.path.exists(train_csv):
            train_csv = _find_under("/kaggle/input", "Train.csv") or train_csv
        if not os.path.exists(test_csv):
            test_csv = _find_under("/kaggle/input", "Test.csv") or test_csv
        if not os.path.isdir(images) or not _glob(os.path.join(images, "*.jpg")):
            cand = _glob("/kaggle/input/**/*.jpg", recursive=True)
            if cand:
                images = os.path.dirname(cand[0])
    return train_csv, test_csv, images

TRAIN_CSV, TEST_CSV, IMAGES_DIR = resolve_paths(CFG)
TRIAL_PRED_DIR = os.path.join(CFG.WORK_DIR, "trial_predictions")
TRIAL_CFG_DIR  = os.path.join(CFG.WORK_DIR, "trial_configs")
TRIAL_RESULTS  = os.path.join(CFG.WORK_DIR, "trial_results.csv")
for _d in (CFG.OUTPUT_DIR, CFG.FINAL_DIR, TRIAL_PRED_DIR, TRIAL_CFG_DIR):
    os.makedirs(_d, exist_ok=True)

print("TRAIN_CSV :", TRAIN_CSV,  "->", os.path.exists(TRAIN_CSV))
print("TEST_CSV  :", TEST_CSV,   "->", os.path.exists(TEST_CSV))
print("IMAGES_DIR:", IMAGES_DIR, "->", os.path.isdir(IMAGES_DIR))
for _m in CFG.MODEL_PATHS:
    print("MODEL     :", _m, "->", os.path.isdir(_m))
assert os.path.exists(TRAIN_CSV), "Train.csv not found"
assert os.path.isdir(IMAGES_DIR), "images dir not found"
assert all(os.path.isdir(m) for m in CFG.MODEL_PATHS), "a model dir is missing"

In [ ]:
# =========================================================
# Cell 5 — text / image utils + augmentation stack
# =========================================================
def clean_label(x):
    return " ".join(str(x).replace("\n", " ").split()).strip()

def clean_output(text: str) -> str:
    text = str(text)
    for tag in ["assistant", "user", "<|assistant|>", "<|user|>"]:
        if tag in text:
            text = text.split(tag)[-1]
    return " ".join(text.split()).strip()

# identical resize ladder for train AND inference (downscale only; the processor then applies MIN/MAX_PIXELS)
def load_image(path):
    img = Image.open(path).convert("RGB")
    w, h = img.size
    aspect = w / h
    if   aspect > 10: img.thumbnail((2048, 384))
    elif aspect > 5:  img.thumbnail((2048, 512))
    elif aspect > 3:  img.thumbnail((1600, 640))
    else:             img.thumbnail((1536, 768))
    return img

# Augmentation magnitudes matched to the full-data forensics of all 6,159 images (FULL_DATA_AUDIT §4.2).
# measured (5th-95th pct)                        original barbados-2              realistic
REALISTIC_AUG = dict(
    AUG_ROTATE_DEG=1.0,  AUG_ROTATE_P=0.5,        # skew -1..+1 deg              | +-4 deg
    AUG_SHEAR_MAX=0.12,  AUG_SHEAR_P=0.4,         # slant spread -0.2..0.8       | 0.15
    AUG_ELASTIC_P=0.3,                            # hand wobble, kept mild       | 0.4
    AUG_HJITTER_FRAC=0.10,                        # x-height varies widely       | 0.12
    AUG_BRIGHT=0.15, AUG_CONTRAST=0.20, AUG_GAMMA=1.3, AUG_PHOTO_P=0.6,   # paper grey 145..216
    AUG_TINT_MAX=0.10, AUG_DESAT_MAX=0.30,        # paper tint R-B 40..71
    AUG_ILLUM_MAX=0.12, AUG_ILLUM_P=0.3,          # illumination CV <= 0.09      | 35 % vignette
    AUG_STROKE_P=0.15, AUG_STROKE_MODE="thicken", # strokes 1.9-3.8 px: 3x3 max-filter erases them
    AUG_BLEED_ALPHA=0.10, AUG_BLEED_P=0.25,       # mid-tone / show-through 3-20 %
    AUG_NOISE_STD=0.012, AUG_NOISE_P=0.3,         # sensor noise sigma 1.8-4.2   | 25.5 grey levels
    AUG_JPEG_QMIN=70, AUG_JPEG_QMAX=92, AUG_JPEG_P=0.3,   # all images q~85     | q40-80
    AUG_FILL="paper",                             # new pixels get the page's own parchment colour, not white
)

def apply_aug_profile(cfg):
    cfg.AUG_FILL = getattr(cfg, "AUG_FILL", "white")
    if getattr(cfg, "AUG_PROFILE", "original") == "realistic":
        for k, v in REALISTIC_AUG.items():
            setattr(cfg, k, v)
    print(f"[aug] profile={cfg.AUG_PROFILE} resample={cfg.AUG_RESAMPLE} rotate=±{cfg.AUG_ROTATE_DEG}° "
          f"noise σ={cfg.AUG_NOISE_STD*255:.1f} jpeg q{cfg.AUG_JPEG_QMIN}-{cfg.AUG_JPEG_QMAX} "
          f"stroke={cfg.AUG_STROKE_MODE}@{cfg.AUG_STROKE_P} illum={cfg.AUG_ILLUM_MAX} fill={cfg.AUG_FILL}")
apply_aug_profile(CFG)

def _paper_grey(gray):
    """mean grey of the pixels above the Otsu threshold (= the parchment), as in FULL_DATA_AUDIT §4."""
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    w = np.cumsum(hist); s = np.cumsum(np.arange(256) * hist); tot, sall = w[-1], s[-1]
    with np.errstate(divide="ignore", invalid="ignore"):
        between = (s * tot - sall * w) ** 2 / (w * (tot - w))
    t = int(np.nanargmax(np.where((w > 0) & (w < tot), between, -1)))
    bg = gray[gray > t]
    return float(bg.mean()) if bg.size else float(gray.mean())

_GROUP_CACHE = {}
def image_group(path):
    """Data group from the ORIGINAL pixels only (no labels) — FULL_DATA_AUDIT §6b:
       B  : height > 150 px            -> 1670s-1710s scans, ~77 px/char, 2x '^', 4x ':'
       A1 : height <= 58 px            -> tight crops, heavy '&' and 'ye'
       A2 : taller, parchment <= 194   -> dark parchment, large hand, almost no '&' / 'ye'
       A3 : taller, parchment  > 194   -> loose crops with neighbour lines, longest lines
    The rule reproduces the label-derived clusters with ~90 % agreement."""
    if path in _GROUP_CACHE:
        return _GROUP_CACHE[path]
    with Image.open(path) as im:
        h = im.size[1]
        if h > CFG.CLASS_HEIGHT_SPLIT:
            g = "B"
        elif h <= CFG.GROUP_A1_MAX_HEIGHT:
            g = "A1"
        else:
            g = "A2" if _paper_grey(np.asarray(im.convert("L"))) <= CFG.GROUP_A2_MAX_PAPER else "A3"
    _GROUP_CACHE[path] = g
    return g

def image_class(path):
    return image_group(path)[0]          # 'A' or 'B'

def key_match(grp, key):
    """does sub-class grp ('A1', ..., 'B') belong to specialist key ('A', 'B', 'A1', ...)?"""
    return grp == key or grp[0] == key

def ocr_prompt(grp=None):
    """CLASS_HINT: append the scan type so one shared model can switch spelling conventions per era."""
    if not (CFG.CLASS_HINT and grp):
        return CFG.OCR_PROMPT
    tag = grp if CFG.HINT_LEVEL == "group" else grp[0]
    return CFG.OCR_PROMPT + f" Scan type: {tag}."

def _paper_color(img):
    """median colour of the brighter half of the page = the parchment (measured background)."""
    a = np.asarray(img.convert("RGB"), np.float32).reshape(-1, 3)
    lum = a.mean(1)
    bg = a[lum >= np.median(lum)]
    return tuple(int(v) for v in np.median(bg, axis=0))

from PIL import ImageOps, ImageFilter
import io as _io
try:
    from scipy.ndimage import gaussian_filter as _gaussian_filter, map_coordinates as _map_coordinates
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

def _photo(arr, rng, b, c, g):
    a = arr / 255.0
    a = np.power(np.clip(a, 0, 1), float(np.exp(rng.uniform(-np.log(g), np.log(g)))))
    a = (a - 0.5) * (1.0 + float(rng.uniform(-c, c))) + 0.5
    a = a * (1.0 + float(rng.uniform(-b, b)))
    return np.clip(a, 0, 1) * 255.0

def _tint(arr, rng, tint_max, desat_max):
    a = (arr / 255.0).copy()
    gray = a.mean(axis=2, keepdims=True)
    d = float(rng.uniform(0, desat_max)); a = a * (1 - d) + gray * d
    a[..., 0] *= 1.0 + float(rng.uniform(0, tint_max))
    a[..., 2] *= 1.0 - float(rng.uniform(0, tint_max))
    return np.clip(a, 0, 1) * 255.0

def _illumination(arr, rng, imax):
    h, w = arr.shape[:2]
    cx, cy = float(rng.uniform(0, w)), float(rng.uniform(0, h))
    yy, xx = np.mgrid[0:h, 0:w]
    d = np.sqrt(((xx - cx) / max(w, 1)) ** 2 + ((yy - cy) / max(h, 1)) ** 2)
    field = (1.0 - float(imax) * (d / (d.max() + 1e-6))).astype(np.float32)[..., None]
    return np.clip(arr * field, 0, 255)

def _bleed(arr, rng, alpha):
    a = float(rng.uniform(0, alpha))
    return np.clip(arr * (1 - a) + arr[:, ::-1, :] * a, 0, 255)

def _elastic(arr, rng, alpha, sigma):
    h, w = arr.shape[:2]
    dx = _gaussian_filter(rng.rand(h, w).astype(np.float32) * 2 - 1, sigma) * alpha
    dy = _gaussian_filter(rng.rand(h, w).astype(np.float32) * 2 - 1, sigma) * alpha
    yy, xx = np.meshgrid(np.arange(h), np.arange(w), indexing="ij")
    coords = [np.clip(yy + dy, 0, h - 1), np.clip(xx + dx, 0, w - 1)]
    out = np.empty_like(arr)
    for ch in range(arr.shape[2]):
        out[..., ch] = _map_coordinates(arr[..., ch], coords, order=1, mode="reflect")
    return out

def augment_image(img, seed=None):
    """Label-preserving photometric + geometric stack, seeded per row (reproducible copies)."""
    if not getattr(CFG, "USE_AUGMENT", False):
        return img
    rng = np.random.RandomState(int(seed) & 0x7fffffff) if seed is not None else np.random
    fill = _paper_color(img) if CFG.AUG_FILL == "paper" else (255, 255, 255)
    if CFG.AUG_ROTATE and rng.rand() < CFG.AUG_ROTATE_P:
        img = img.rotate(float(rng.uniform(-CFG.AUG_ROTATE_DEG, CFG.AUG_ROTATE_DEG)),
                         resample=Image.BICUBIC, expand=True, fillcolor=fill)
    if CFG.AUG_SHEAR and rng.rand() < CFG.AUG_SHEAR_P:
        sh = float(rng.uniform(-CFG.AUG_SHEAR_MAX, CFG.AUG_SHEAR_MAX)); w, h = img.size
        xshift = abs(sh) * h
        img = img.transform((w + int(round(xshift)), h), Image.AFFINE,
                            (1, sh, -xshift if sh > 0 else 0.0, 0, 1, 0),
                            resample=Image.BICUBIC, fillcolor=fill)
    if CFG.AUG_STROKE and rng.rand() < CFG.AUG_STROKE_P:
        # dark ink on light paper: MinFilter = thicker strokes (ink spread), MaxFilter = thinner (can erase)
        thin = CFG.AUG_STROKE_MODE == "both" and rng.rand() >= 0.5
        img = img.filter(ImageFilter.MaxFilter(3) if thin else ImageFilter.MinFilter(3))
    if CFG.AUG_HJITTER and rng.rand() < CFG.AUG_HJITTER_P:
        w, h = img.size
        f = float(rng.uniform(1 - CFG.AUG_HJITTER_FRAC, 1 + CFG.AUG_HJITTER_FRAC))
        img = img.resize((w, max(1, int(round(h * f)))), Image.BICUBIC)
    if CFG.AUG_PAD and rng.rand() < CFG.AUG_PAD_P:
        w, h = img.size; pf = CFG.AUG_PAD_FRAC
        img = ImageOps.expand(img, (int(rng.uniform(0, pf) * w), int(rng.uniform(0, pf) * h),
                                    int(rng.uniform(0, pf) * w), int(rng.uniform(0, pf) * h)),
                              fill=fill)
    arr = np.asarray(img.convert("RGB"), np.float32)
    if CFG.AUG_ELASTIC and _HAS_SCIPY and rng.rand() < CFG.AUG_ELASTIC_P:
        arr = _elastic(arr, rng, CFG.AUG_ELASTIC_ALPHA, CFG.AUG_ELASTIC_SIGMA)
    if CFG.AUG_PHOTO and rng.rand() < CFG.AUG_PHOTO_P:
        arr = _photo(arr, rng, CFG.AUG_BRIGHT, CFG.AUG_CONTRAST, CFG.AUG_GAMMA)
    if CFG.AUG_TINT and rng.rand() < CFG.AUG_TINT_P:
        arr = _tint(arr, rng, CFG.AUG_TINT_MAX, CFG.AUG_DESAT_MAX)
    if CFG.AUG_ILLUM and rng.rand() < CFG.AUG_ILLUM_P:
        arr = _illumination(arr, rng, CFG.AUG_ILLUM_MAX)
    if CFG.AUG_BLEED and rng.rand() < CFG.AUG_BLEED_P:
        arr = _bleed(arr, rng, CFG.AUG_BLEED_ALPHA)
    if CFG.AUG_NOISE and rng.rand() < CFG.AUG_NOISE_P:
        arr = arr + rng.randn(*arr.shape).astype(np.float32) * (CFG.AUG_NOISE_STD * 255.0)
    out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    if CFG.AUG_JPEG and rng.rand() < CFG.AUG_JPEG_P:
        buf = _io.BytesIO()
        out.save(buf, format="JPEG", quality=int(rng.randint(CFG.AUG_JPEG_QMIN, CFG.AUG_JPEG_QMAX + 1)))
        buf.seek(0); out = Image.open(buf).convert("RGB")
    return out

def resolve_image(base_dir, image_id):
    exact = os.path.join(base_dir, f"{image_id}.jpg")
    if os.path.exists(exact):
        return exact
    matches = _glob(os.path.join(base_dir, f"{image_id}*.jpg"))
    return matches[0] if matches else None

def model_short_name(msrc):
    parts = [p for p in msrc.replace("\\", "/").strip("/").split("/") if p]
    return (re.sub(r"[^A-Za-z0-9]+", "_", "_".join(parts[-4:]))[:60] or "model")

def legacy_short_name(msrc):
    """adapter dir name written by the ORIGINAL barbados-2.ipynb (last 3 path parts)."""
    parts = [p for p in msrc.replace("\\", "/").strip("/").split("/") if p]
    return (re.sub(r"[^A-Za-z0-9]+", "_", "_".join(parts[-3:]))[:48] or "model")

def model_size_tag(name):
    m = re.search(r"(?<![0-9])(\d+b)(?![0-9])", name.lower())
    return m.group(1) if m else name[-12:]

def free_cuda(*names):
    for n in names:
        if n in globals():
            del globals()[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

In [ ]:
# =========================================================
# Cell 6 — model & processor loaders (pixel limits enforced + verified)
# =========================================================
def load_vlm_model(path, dtype, device_map=None):
    base = {"trust_remote_code": True}
    if device_map is not None:
        base["device_map"] = device_map
    if getattr(CFG, "ATTN_IMPL", None):
        base["attn_implementation"] = CFG.ATTN_IMPL
    for dkey in ("dtype", "torch_dtype"):
        try:
            return AutoVLM.from_pretrained(path, **{dkey: dtype}, **base)
        except TypeError:
            continue
        except Exception:
            if "attn_implementation" in base:
                base.pop("attn_implementation")
                try:
                    return AutoVLM.from_pretrained(path, **{dkey: dtype}, **base)
                except TypeError:
                    continue
            raise
    return AutoVLM.from_pretrained(path, **base)

def _repair_preprocessor(path):
    dst = os.path.join(CFG.WORK_DIR, "_patched_" + os.path.basename(path.rstrip("/\\")))
    if os.path.isdir(dst):
        shutil.rmtree(dst)
    def _ignore_weights(_d, names):
        return [n for n in names if n.endswith(
            (".safetensors", ".bin", ".pt", ".pth", ".gguf", ".h5", ".msgpack", ".onnx"))]
    shutil.copytree(path, dst, ignore=_ignore_weights)
    pcfg = os.path.join(dst, "preprocessor_config.json")
    cfg = {}
    if os.path.exists(pcfg):
        with open(pcfg, "r") as f:
            cfg = json.load(f)
    prefer = ["Qwen2VLImageProcessorFast", "Qwen2_5_VLImageProcessorFast", "Qwen3VLImageProcessorFast",
              "Qwen2VLImageProcessor", "Qwen2_5_VLImageProcessor", "Qwen3VLImageProcessor"]
    avail = [c for c in prefer if hasattr(transformers, c)]
    if avail:
        cfg["image_processor_type"] = avail[0]
    else:
        cfg.pop("image_processor_type", None)
    with open(pcfg, "w") as f:
        json.dump(cfg, f)
    return dst

def _smart_resize(h, w, factor, min_pixels, max_pixels):
    h_bar, w_bar = round(h / factor) * factor, round(w / factor) * factor
    if h_bar * w_bar > max_pixels:
        beta = math.sqrt((h * w) / max_pixels)
        h_bar = max(factor, math.floor(h / beta / factor) * factor)
        w_bar = max(factor, math.floor(w / beta / factor) * factor)
    elif h_bar * w_bar < min_pixels:
        beta = math.sqrt(min_pixels / (h * w))
        h_bar, w_bar = math.ceil(h * beta / factor) * factor, math.ceil(w * beta / factor) * factor
    return h_bar, w_bar

def _force_pixel_limits(ip, min_pixels, max_pixels):
    """transformers 5.0 honours ip.min_pixels/ip.max_pixels attributes; >=5.1x only reads ip.size.
    Set every representation so the limits hold on any version."""
    try:
        sz = dict(ip.size) if not isinstance(ip.size, dict) else dict(ip.size)
    except Exception:
        sz = {}
    sz = {k: v for k, v in sz.items() if v is not None}
    sz["shortest_edge"], sz["longest_edge"] = int(min_pixels), int(max_pixels)
    try:
        ip.size = sz
    except Exception:
        pass
    for k, v in (("min_pixels", min_pixels), ("max_pixels", max_pixels)):
        try:
            setattr(ip, k, int(v))
        except Exception:
            pass

def load_vlm_processor(path, max_pixels=None, min_pixels=None):
    kw_px = {}
    if max_pixels is not None and min_pixels is not None:
        kw_px = dict(min_pixels=int(min_pixels), max_pixels=int(max_pixels))
    attempts = [dict(use_fast=True), dict(use_fast=False),
                dict(use_fast=True, trust_remote_code=True), dict(use_fast=False, trust_remote_code=True)]
    proc, last = None, None
    for src in (path, None):
        if src is None:
            print("[processor] direct load failed; repairing. cause:", repr(last)[:200])
            src = _repair_preprocessor(path)
        for kw in attempts:
            for extra in (kw_px, {}):
                try:
                    proc = AutoProcessor.from_pretrained(src, **kw, **extra); break
                except Exception as e:
                    last = e
            if proc is not None: break
        if proc is not None: break
    if proc is None:
        raise last
    ip = proc.image_processor
    if kw_px:
        _force_pixel_limits(ip, min_pixels, max_pixels)
    # ---- verify on a probe strip: the grid must match smart_resize(min/max) ----
    probe = Image.new("RGB", (1060, 57), (220, 200, 170))
    fac = int(getattr(ip, "patch_size", 14)) * int(getattr(ip, "merge_size", 2))
    out = ip(images=[probe], return_tensors="pt")
    g = out["image_grid_thw"][0].tolist()
    got = (g[1] * ip.patch_size, g[2] * ip.patch_size)
    if kw_px:
        exp = _smart_resize(57, 1060, fac, min_pixels, max_pixels)
        assert tuple(got) == tuple(exp), f"pixel limits NOT applied: processor gives {got}, expected {exp}"
    print(f"[processor] {type(proc).__name__} | {type(ip).__name__} | probe 1060x57 -> {got[1]}x{got[0]} px, "
          f"{g[1]*g[2]//(ip.merge_size**2)} visual tokens")
    return proc

In [ ]:
# =========================================================
# Cell 7 — OFFICIAL metric, MBR / ROVER ensembling, error forensics, paired bootstrap
#   score = 0.5*(1 - word_edits_per_line/12) + 0.5*(1 - char_edits_per_line/55)   (higher = better)
# =========================================================
WORD_XMAX, CHAR_XMAX = 12.0, 55.0

def norm_text(s):
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    return " ".join(str(s).split())

def line_edits(ref, hyp):
    r, h = norm_text(ref), norm_text(hyp)
    return _lev(r.split(), h.split()), _lev(r, h)

def line_cost(ref, hyp):
    w, c = line_edits(ref, hyp)
    return 0.5 * w / WORD_XMAX + 0.5 * c / CHAR_XMAX

def official_metric(refs, hyps):
    ed = np.array([line_edits(r, h) for r, h in zip(refs, hyps)], float).reshape(-1, 2)
    we, ce = ed[:, 0], ed[:, 1]
    nw = max(1, sum(len(norm_text(r).split()) for r in refs)); nc = max(1, sum(len(norm_text(r)) for r in refs))
    return {"score": float(0.5 * (1 - we.mean() / WORD_XMAX) + 0.5 * (1 - ce.mean() / CHAR_XMAX)),
            "word_edits_per_line": float(we.mean()), "char_edits_per_line": float(ce.mean()),
            "wer_corpus": float(we.sum() / nw), "cer_corpus": float(ce.sum() / nc),
            "n": len(refs), "empty": int(sum(not norm_text(h) for h in hyps))}

def mbr_select(cands, weights=None, return_risk=False):
    """Minimum-Bayes-risk pick under the official per-line cost. The minimum expected cost ('risk')
    doubles as a confidence: low risk = the candidate pool agrees."""
    cands = [norm_text(c) for c in cands]
    weights = [1.0] * len(cands) if weights is None else weights
    agg = collections.OrderedDict()
    for c, w in zip(cands, weights):
        agg[c] = agg.get(c, 0.0) + float(w)
    uniq = list(agg)
    if len(uniq) == 1:
        return (uniq[0], 0.0) if return_risk else uniq[0]
    wv = np.array([agg[u] for u in uniq]); wv = wv / wv.sum()
    risks = [sum(w * line_cost(c, h) for c, w in zip(uniq, wv)) for h in uniq]
    k = int(np.argmin(risks))
    return (uniq[k], float(risks[k])) if return_risk else uniq[k]

def nbest_posteriors(total_logprobs, temperature=1.0):
    s = np.asarray(total_logprobs, float) / max(temperature, 1e-6)
    s = s - s.max(); p = np.exp(s)
    return p / p.sum()

def _align(cons, hyp):
    n, m = len(cons), len(hyp)
    D = np.zeros((n + 1, m + 1), np.int32); D[:, 0] = np.arange(n + 1); D[0, :] = np.arange(m + 1)
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            D[i, j] = min(D[i-1, j-1] + (cons[i-1] != hyp[j-1]), D[i-1, j] + 1, D[i, j-1] + 1)
    i, j, out = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and D[i, j] == D[i-1, j-1] + (cons[i-1] != hyp[j-1]):
            out.append((i - 1, hyp[j - 1])); i -= 1; j -= 1
        elif i > 0 and D[i, j] == D[i-1, j] + 1:
            out.append((i - 1, None)); i -= 1
        else:
            out.append((None, hyp[j - 1])); j -= 1
    return out[::-1]

def rover(hyps, weights=None, level="word"):
    """ROVER: align every system's top-1 into a confusion network, weighted vote per slot
    (an absent system votes for deletion). level='word' matches the metric's 4.58x word premium."""
    split = (lambda s: norm_text(s).split()) if level == "word" else (lambda s: list(norm_text(s)))
    join = " ".join if level == "word" else "".join
    seqs = [split(h) for h in hyps]
    weights = [1.0] * len(seqs) if weights is None else list(weights)
    order = list(np.argsort(-np.asarray(weights), kind="stable"))
    seqs = [seqs[k] for k in order]; weights = [weights[k] for k in order]
    slots = [{0: t} for t in seqs[0]]
    for s in range(1, len(seqs)):
        cons = [collections.Counter(sl.values()).most_common(1)[0][0] if sl else "" for sl in slots]
        new = []
        for ci, t in _align(cons, seqs[s]):
            if ci is None:
                new.append({s: t})
            else:
                sl = dict(slots[ci])
                if t is not None:
                    sl[s] = t
                new.append(sl)
        slots = new
    out = []
    for sl in slots:
        agg = collections.Counter()
        for s, t in sl.items():
            agg[t] += weights[s]
        agg[None] += sum(weights[s] for s in range(len(seqs)) if s not in sl)
        best = max(agg.items(), key=lambda kv: kv[1])[0]
        if best is not None:
            out.append(best)
    return norm_text(join(out))

def paired_delta(refs, hyp_a, hyp_b, n_boot=10000, seed=0):
    """challenger B vs champion A — paired bootstrap over whole lines."""
    la = np.array([line_cost(r, h) for r, h in zip(refs, hyp_a)])
    lb = np.array([line_cost(r, h) for r, h in zip(refs, hyp_b)])
    d = la - lb
    rng = np.random.default_rng(seed)
    boots = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    ea = np.array([line_edits(r, h) for r, h in zip(refs, hyp_a)]); eb = np.array([line_edits(r, h) for r, h in zip(refs, hyp_b)])
    return {"delta": float(d.mean()), "ci95_lo": float(np.percentile(boots, 2.5)), "ci95_hi": float(np.percentile(boots, 97.5)),
            "p_better": float((boots > 0).mean()), "improved": int((d > 1e-12).sum()), "worsened": int((d < -1e-12).sum()),
            "word_edits_saved": int(ea[:, 0].sum() - eb[:, 0].sum()), "char_edits_saved": int(ea[:, 1].sum() - eb[:, 1].sum())}

def _word_err_class(r, h):
    if r.lower() == h.lower(): return "casing"
    strip = lambda x: "".join(ch for ch in x if ch.isalnum())
    if strip(r) == strip(h): return "punct/markup"
    if strip(r).lower() == strip(h).lower(): return "casing+punct"
    d = _lev(r, h)
    return "1-char" if d == 1 else "2-char" if d == 2 else "3+char"

def error_forensics(refs, hyps, top=20):
    try:
        from rapidfuzz.distance import Levenshtein as L
    except Exception:
        return {"note": "rapidfuzz unavailable"}
    ops, csub, wsub, wcls = collections.Counter(), collections.Counter(), collections.Counter(), collections.Counter()
    cdel, cins, wdel, wins = collections.Counter(), collections.Counter(), collections.Counter(), collections.Counter()
    for r, h in zip(refs, hyps):
        r, h = norm_text(r), norm_text(h); rw, hw = r.split(), h.split()
        for op in L.editops(rw, hw):
            ops["word_" + op.tag] += 1
            if op.tag == "replace":
                wsub[f"{rw[op.src_pos]} -> {hw[op.dest_pos]}"] += 1; wcls[_word_err_class(rw[op.src_pos], hw[op.dest_pos])] += 1
            elif op.tag == "delete": wdel[rw[op.src_pos]] += 1
            else: wins[hw[op.dest_pos]] += 1
        for op in L.editops(r, h):
            ops["char_" + op.tag] += 1
            if op.tag == "replace": csub[f"{r[op.src_pos]!r}->{h[op.dest_pos]!r}"] += 1
            elif op.tag == "delete": cdel[repr(r[op.src_pos])] += 1
            else: cins[repr(h[op.dest_pos])] += 1
    return {"ops": dict(ops), "word_sub_classes": dict(wcls), "word_sub": wsub.most_common(top),
            "word_del": wdel.most_common(top), "word_ins": wins.most_common(top),
            "char_sub": csub.most_common(top), "char_del": cdel.most_common(top), "char_ins": cins.most_common(top)}

def score_by_group(refs, hyps, groups):
    g = np.asarray(groups); out = {}
    for k in sorted(set(g.tolist()), key=str):
        idx = np.where(g == k)[0]
        m = official_metric([refs[i] for i in idx], [hyps[i] for i in idx])
        out[str(k)] = f"n={len(idx):4d} score={m['score']:.4f} w/line={m['word_edits_per_line']:.3f} c/line={m['char_edits_per_line']:.3f}"
    return out

# sanity checks
assert line_edits("a b c", "") == (3, 5)
assert abs(official_metric(["a b c"], [""])["score"] - (0.5 * (1 - 3/12) + 0.5 * (1 - 5/55))) < 1e-12
assert rover(["the said John", "the sd John", "the said Jon"]) == "the said John"
assert mbr_select(["a b c", "a b c", "a x c"]) == "a b c"
print("official metric + ensemble helpers OK")

In [ ]:
# =========================================================
# Cell 8 — samples, Fold-0 split (identical IDs to barbados-2), repaired protocol
# =========================================================
def build_samples(df, base_dir):
    corrupt = set(CFG.CORRUPT_IDS)
    samples, missing, excluded = [], 0, 0
    for _, row in df.iterrows():
        image_id, label = str(row["ID"]).strip(), clean_label(row["Target"])
        if not image_id or not label:
            continue
        if image_id in corrupt:
            excluded += 1; continue
        path = resolve_image(base_dir, image_id)
        if path is None:
            missing += 1; continue
        samples.append({"id": image_id, "image": path, "text": label})
    print(f"[DATASET] found={len(samples)}, missing={missing}, excluded_corrupt={excluded}")
    return samples

df = pd.read_csv(TRAIN_CSV).sample(frac=1, random_state=CFG.SEED).reset_index(drop=True)
all_samples = build_samples(df, IMAGES_DIR)
FOLDS = list(KFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(all_samples))

if CFG.RUN_MODE == "fold0":
    tr_idx, va_idx = FOLDS[CFG.FOLD]
else:                                   # "full": everything is training data, no validation
    tr_idx, va_idx = np.arange(len(all_samples)), np.array([], int)
_rng = np.random.RandomState(CFG.SEED + 1000 + CFG.FOLD)
_perm = _rng.permutation(tr_idx)
n_inner = int(round(CFG.INNER_CKPT_FRAC * len(_perm))) if CFG.CKPT_SELECTION == "inner_loss" else 0
inner_idx, core_idx = _perm[:n_inner], _perm[n_inner:]
core_samples  = [all_samples[i] for i in core_idx]
inner_samples = [all_samples[i] for i in inner_idx]
val_samples   = [all_samples[i] for i in va_idx]
VAL_IDS  = [s["id"] for s in val_samples]
VAL_GT   = [s["text"] for s in val_samples]
VAL_PATHS = [s["image"] for s in val_samples]
assert not (set(VAL_IDS) & {s["id"] for s in core_samples + inner_samples}), "validation leak!"
print(f"mode={CFG.RUN_MODE} fold={CFG.FOLD}: train={len(core_samples)} inner_ckpt={len(inner_samples)} "
      f"val={len(val_samples)} (ALL scored)")
# data group of every image (A1/A2/A3 = 1630s-60s scans, B = 1670s-1710s high-res scans) — from pixels only
for s in all_samples:
    s["grp"] = image_group(s["image"])
VAL_GROUP = [s["grp"] for s in val_samples]
VAL_CLASS = [g[0] for g in VAL_GROUP]
VAL_LEN_BUCKET = [("<50" if len(t) < 50 else "50-69" if len(t) < 70 else "70+") for t in VAL_GT]
_cc = lambda ss: dict(sorted(collections.Counter(s["grp"] for s in ss).items()))
print(f"group balance  train={_cc(core_samples)}  inner={_cc(inner_samples)}  val={_cc(val_samples)}")

_test_df = pd.read_csv(TEST_CSV, encoding="utf-8-sig")
_test_df.columns = [c.lstrip("\ufeff") for c in _test_df.columns]
TEST_IDS   = [str(r["ID"]).strip() for r in _test_df.to_dict("records")]
TEST_PATHS = [os.path.join(IMAGES_DIR, f"{rid}.jpg") for rid in TEST_IDS]
print("test rows:", len(TEST_IDS), "| missing test images:", sum(not os.path.exists(p) for p in TEST_PATHS))
_known = set(TEST_IDS) | {str(i).strip() for i in pd.read_csv(TRAIN_CSV).ID}
UNL_IDS = sorted(os.path.basename(p)[:-4] for p in _glob(os.path.join(IMAGES_DIR, "*.jpg"))
                 if os.path.basename(p)[:-4] not in _known)
UNL_PATHS = [os.path.join(IMAGES_DIR, f"{i}.jpg") for i in UNL_IDS]
print("unlisted images (in images/, in neither CSV):", len(UNL_IDS))
TEST_GROUP = [image_group(p) if os.path.exists(p) else "A1" for p in TEST_PATHS]
UNL_GROUP  = [image_group(p) for p in UNL_PATHS]
TEST_CLASS = [g[0] for g in TEST_GROUP]
print("group balance  test=", dict(sorted(collections.Counter(TEST_GROUP).items())),
      " unlisted=", dict(sorted(collections.Counter(UNL_GROUP).items())))

# ---- optional pseudo-labels (test + unlisted) from a previous session's ensemble ----
pseudo_samples = []
if CFG.PSEUDO_LABEL_CSV and os.path.exists(CFG.PSEUDO_LABEL_CSV):
    _pl = pd.read_csv(CFG.PSEUDO_LABEL_CSV)
    _pl = _pl[_pl.Target.fillna("").str.split().str.len().between(3, 30)]        # drop empty / runaway lines
    _pl = _pl[_pl.ID.map(lambda i: os.path.exists(os.path.join(IMAGES_DIR, f"{i}.jpg")))]
    _pl["grp"] = [image_group(os.path.join(IMAGES_DIR, f"{i}.jpg")) for i in _pl.ID]
    if "keep_for_training" in _pl:  # file from work/08_build_pseudo_labels.py: calibrated per-class selection already made
        _pl = _pl[_pl.keep_for_training.astype(str).str.lower().isin(["true", "1"])]
    elif CFG.PSEUDO_PER_CLASS:      # class B is harder -> higher risk; a global cut would drop most of it
        _q = _pl.groupby(_pl.grp.str[0]).risk.transform(lambda r: r.quantile(CFG.PSEUDO_KEEP_FRAC))
        _pl = _pl[_pl.risk <= _q]
    else:
        _pl = _pl[_pl.risk <= _pl.risk.quantile(CFG.PSEUDO_KEEP_FRAC)]
    pseudo_samples = [{"id": r.ID, "image": os.path.join(IMAGES_DIR, f"{r.ID}.jpg"), "text": clean_label(r.Target),
                       "grp": r.grp} for r in _pl.itertuples()]
    assert not ({s["id"] for s in pseudo_samples} & set(VAL_IDS)), "pseudo-labels must not cover validation lines"
    print(f"pseudo-labelled lines added to TRAIN: {len(pseudo_samples)} (risk <= q{CFG.PSEUDO_KEEP_FRAC:.2f}"
          f"{' per class' if CFG.PSEUDO_PER_CLASS else ''}) by group:", dict(collections.Counter(_pl.grp)))
pd.DataFrame({"ID": VAL_IDS}).to_csv(os.path.join(CFG.WORK_DIR, f"fold{CFG.FOLD}_val_ids.csv"), index=False)

In [ ]:
# =========================================================
# Cell 9 — collator with assistant-boundary label masking (masking unchanged from barbados-2)
# =========================================================
MASK_FAIL_COUNT = 0
_AUG_VISITS = collections.Counter()

def _aug_seed(row_seed):
    """AUG_RESAMPLE: a new, still reproducible, augmentation every time a line is seen.
    Mixes the per-process visit count (num_workers=0) with the DataLoader worker seed, which
    PyTorch re-draws every epoch from the Trainer-seeded generator (num_workers>0)."""
    if not CFG.AUG_RESAMPLE:
        return row_seed
    v = _AUG_VISITS[row_seed]; _AUG_VISITS[row_seed] += 1
    return (int(row_seed) * 1000003 + v * 7919 + int(torch.initial_seed()) % 1000000007) & 0x7fffffff

def collate_fn(examples, processor):
    global MASK_FAIL_COUNT
    images, texts, classes = [], [], []
    for ex in examples:
        if os.path.exists(ex["image"]):
            img = load_image(ex["image"])
            if CFG.USE_AUGMENT and ex.get("aug", False):
                img = augment_image(img, _aug_seed(ex.get("seed")))
            images.append(img); texts.append(ex["text"]); classes.append(ex.get("grp"))
    if len(images) == 0:
        return None
    tokenizer = processor.tokenizer
    full_msgs, prompt_msgs = [], []
    for img, label, cls in zip(images, texts, classes):
        user_turn = {"role": "user", "content": [{"type": "text", "text": ocr_prompt(cls)},
                                                 {"type": "image", "image": img}]}
        full_msgs.append([user_turn, {"role": "assistant", "content": [{"type": "text", "text": label}]}])
        prompt_msgs.append([user_turn])
    texts_out = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=False) for m in full_msgs]
    batch = processor(text=texts_out, images=images, padding=True, truncation=True, return_tensors="pt")
    labels = batch["input_ids"].clone()
    for i, pm in enumerate(prompt_msgs):
        try:
            prompt_txt = processor.apply_chat_template(pm, tokenize=False, add_generation_prompt=True)
            prompt_ids = processor(text=[prompt_txt], images=[images[i]], return_tensors="pt", truncation=True)["input_ids"][0]
            n_prompt = int((prompt_ids != tokenizer.pad_token_id).sum().item())
            labels[i, :n_prompt] = -100
        except Exception:
            MASK_FAIL_COUNT += 1
    labels[labels == tokenizer.pad_token_id] = -100
    image_token = getattr(processor, "image_token", None) or "<|image_pad|>"
    image_token_id = tokenizer.convert_tokens_to_ids(image_token)
    if isinstance(image_token_id, int) and image_token_id >= 0:
        labels[labels == image_token_id] = -100
    batch["labels"] = labels
    return batch

In [ ]:
# =========================================================
# Cell 10 — batched N-best generation with sequence scores
#   returns, per image, a list of (text, total_logprob, n_tokens, truncated) sorted best-first
# =========================================================
def _gen_config(model, var):
    gc_ = copy.deepcopy(model.generation_config)
    gc_.do_sample = False
    for k in ("temperature", "top_p", "top_k"):
        try: setattr(gc_, k, None)
        except Exception: pass
    nb = int(var.get("num_beams", 1)); nr = min(int(var.get("n_return", 1)), nb)
    gc_.num_beams = nb
    gc_.num_return_sequences = nr
    gc_.max_new_tokens = CFG.MAX_NEW_TOKENS
    gc_.repetition_penalty = float(var.get("repetition_penalty", 1.0))
    gc_.no_repeat_ngram_size = int(var.get("no_repeat_ngram_size", 0))
    gc_.length_penalty = float(var.get("length_penalty", 1.0))
    gc_.early_stopping = True if nb > 1 else False
    gc_.output_scores = True
    gc_.return_dict_in_generate = True
    return gc_, nb, nr

def generate_nbest(model, processor, images, var, classes=None):
    classes = classes or [None] * len(images)
    prompts = [processor.apply_chat_template(
        [{"role": "user", "content": [{"type": "text", "text": ocr_prompt(c)}, {"type": "image", "image": im}]}],
        tokenize=False, add_generation_prompt=True) for im, c in zip(images, classes)]
    tok = processor.tokenizer
    old_side = tok.padding_side; tok.padding_side = "left"
    try:
        inputs = processor(text=prompts, images=images, return_tensors="pt", padding=True)
    finally:
        tok.padding_side = old_side
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    gcfg, nb, nr = _gen_config(model, var)
    with torch.inference_mode():
        out = model.generate(**inputs, generation_config=gcfg)
    P = inputs["input_ids"].shape[1]
    gen = out.sequences[:, P:]
    stop_ids = {tok.pad_token_id, tok.eos_token_id, tok.convert_tokens_to_ids("<|im_end|>")}
    stop_ids = {s for s in stop_ids if isinstance(s, int) and s >= 0}
    is_stop = torch.zeros_like(gen, dtype=torch.bool)
    for s in stop_ids:
        is_stop |= gen == s
    has_stop = is_stop.any(1)
    first_stop = torch.where(has_stop, is_stop.float().argmax(1), torch.full_like(has_stop, gen.shape[1], dtype=torch.long))
    n_tok = (first_stop + has_stop.long()).clamp(min=1)            # generated tokens incl. EOS
    if nb > 1 and getattr(out, "sequences_scores", None) is not None:
        lp = float(gcfg.length_penalty)
        total = out.sequences_scores.float() * n_tok.float() ** lp  # undo length normalisation
    else:
        ts = model.compute_transition_scores(out.sequences, out.scores, normalize_logits=True)
        valid = torch.arange(ts.shape[1], device=ts.device)[None, :] < n_tok[:, None].to(ts.device)
        total = torch.where(valid & torch.isfinite(ts), ts, torch.zeros_like(ts)).sum(1)
    texts = [clean_output(t) for t in processor.batch_decode(gen, skip_special_tokens=True)]
    res = []
    for b in range(len(images)):
        rows = []
        for r in range(nr):
            k = b * nr + r
            rows.append((texts[k], float(total[k]), int(n_tok[k]), bool(not has_stop[k])))
        res.append(rows)
    return res

def transcribe(model, processor, paths, var, desc=None):
    """-> list (aligned with paths) of N-best lists. Missing images -> [("", -inf, 0, False)]."""
    bs = max(1, int(CFG.BATCH_SIZE) // max(1, int(var.get("num_beams", 1)) // 3))
    out = [[("", float("-inf"), 0, False)] for _ in paths]
    idxs = [i for i, p in enumerate(paths) if p and os.path.exists(p)]
    for b in tqdm(range(0, len(idxs), bs), desc=desc, leave=False):
        chunk = idxs[b:b + bs]
        imgs = [load_image(paths[i]) for i in chunk]
        cls = [image_group(paths[i]) for i in chunk] if CFG.CLASS_HINT else None
        try:
            res = generate_nbest(model, processor, imgs, var, cls)
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            res = [generate_nbest(model, processor, [im], var, [c] if cls else None)[0]
                   for im, c in zip(imgs, cls or [None] * len(imgs))]
        for i, r in zip(chunk, res):
            out[i] = r
    return out

In [ ]:
# =========================================================
# Cell 11 — trial harness: persistence, champion tracking, standard report block
# =========================================================
def _prev_results():
    frames = []
    for d in [CFG.WORK_DIR] + list(CFG.PREV_RESULTS_DIRS):
        p = os.path.join(d, "trial_results.csv")
        if os.path.exists(p):
            try: frames.append(pd.read_csv(p))
            except Exception: pass
    return pd.concat(frames, ignore_index=True).drop_duplicates("trial") if frames else pd.DataFrame()

def champion_row():
    r = _prev_results()
    if len(r) == 0 or "score" not in r: return None
    r = r[r.status.isin(["PROMOTED", "BASELINE", "EVALUATED"]) & r.score.notna()]
    return None if len(r) == 0 else r.sort_values("score", ascending=False).iloc[0]

def save_predictions(trial, ids, gts, nbests, chosen=None, split="val"):
    rows = []
    for k, i in enumerate(ids):
        nb = nbests[k]
        top = chosen[k] if chosen is not None else nb[0][0]
        rows.append({"ID": i, "GroundTruth": gts[k] if gts is not None else None, "pred": top,
                     "logprob": nb[0][1] if nb else None, "n_tokens": nb[0][2] if nb else None,
                     "truncated": nb[0][3] if nb else None,
                     "nbest": json.dumps([[t, round(s, 4)] for t, s, _, _ in nb], ensure_ascii=False)})
    p = os.path.join(TRIAL_PRED_DIR, f"{trial}__{split}.csv")
    pd.DataFrame(rows).to_csv(p, index=False)
    return p

def report_trial(trial, desc, model, variant, hyps, runtime_s, n_train, trunc=0, baseline_hyps=None, extra=None):
    m = official_metric(VAL_GT, hyps)
    base = paired_delta(VAL_GT, baseline_hyps, hyps) if baseline_hyps is not None else None
    ch = champion_row()
    ch_score = float(ch.score) if ch is not None else float("nan")
    peak = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
    print("=" * 66)
    print(f"TRIAL: {trial}\nDESCRIPTION: {desc}\nMODEL: {model} | DECODE: {variant}")
    print("=" * 66)
    print(f"Official validation score : {m['score']:.5f}")
    print(f"Word edits / line (LB WER) : {m['word_edits_per_line']:.4f}   corpus WER {m['wer_corpus']:.4f}")
    print(f"Char edits / line (LB CER) : {m['char_edits_per_line']:.4f}   corpus CER {m['cer_corpus']:.4f}")
    if base is not None:
        print(f"Delta vs baseline          : {base['delta']:+.5f}  CI95 [{base['ci95_lo']:+.5f}, {base['ci95_hi']:+.5f}]"
              f"  P(better)={base['p_better']:.3f}  improved={base['improved']} worsened={base['worsened']}"
              f"  word/char edits saved={base['word_edits_saved']}/{base['char_edits_saved']}")
    print(f"Delta vs champion          : {m['score'] - ch_score:+.5f}  (champion={ch_score:.5f})")
    print(f"Runtime                    : {runtime_s/60:.1f} min   Peak GPU RAM : {peak:.1f} GB")
    print(f"Train examples             : {n_train}   Validation examples : {m['n']}")
    print(f"Empty predictions          : {m['empty']}   Truncated predictions : {trunc}")
    print("By class (A<=150px, B>150px):", score_by_group(VAL_GT, hyps, VAL_CLASS))
    print("By group (A1/A2/A3/B)      :", score_by_group(VAL_GT, hyps, VAL_GROUP))
    print("By label length          :", score_by_group(VAL_GT, hyps, VAL_LEN_BUCKET))
    print("=" * 66)
    row = {"trial": trial, "session": TRIAL_ID, "description": desc, "model": model, "variant": variant,
           "score": m["score"], "word_edits_per_line": m["word_edits_per_line"], "char_edits_per_line": m["char_edits_per_line"],
           "wer_corpus": m["wer_corpus"], "cer_corpus": m["cer_corpus"],
           "delta_baseline": base["delta"] if base else np.nan, "delta_ci95_lo": base["ci95_lo"] if base else np.nan,
           "delta_ci95_hi": base["ci95_hi"] if base else np.nan, "p_better_baseline": base["p_better"] if base else np.nan,
           "delta_champion": m["score"] - ch_score, "runtime_min": runtime_s / 60, "peak_gpu_gb": peak,
           "n_train": n_train, "n_val": m["n"], "empty": m["empty"], "truncated": trunc,
           "status": "EVALUATED", "time": time.strftime("%Y-%m-%d %H:%M:%S")}
    for g in ("A", "B"):                                   # per-class score columns for the trial table
        idx = [i for i, c in enumerate(VAL_CLASS) if c == g]
        row[f"score_{g}"] = official_metric([VAL_GT[i] for i in idx], [hyps[i] for i in idx])["score"] if idx else np.nan
    row.update({"lr": CFG.LEARNING_RATE, "aug_profile": CFG.AUG_PROFILE, "class_hint": CFG.CLASS_HINT,
                "class_aug_copies": json.dumps(CFG.CLASS_AUG_COPIES)})
    if extra: row.update(extra)
    prev = pd.read_csv(TRIAL_RESULTS) if os.path.exists(TRIAL_RESULTS) else pd.DataFrame()
    pd.concat([prev, pd.DataFrame([row])], ignore_index=True).to_csv(TRIAL_RESULTS, index=False)
    with open(os.path.join(TRIAL_CFG_DIR, f"{trial}.json"), "w") as f:
        json.dump({"trial": trial, "session": TRIAL_ID, "description": desc, "model": model, "variant": variant,
                   "decode": CFG.DECODE_VARIANTS.get(variant), "cfg": {k: (list(v) if isinstance(v, (set, tuple)) else v)
                   for k, v in TRIAL_CONFIG.items()}}, f, indent=1, default=str)
    return m

In [ ]:
# =========================================================
# Cell 12 — MEGA setup: logging, prompts per scheme, systems, training rows, step validation callback
# =========================================================
import torch.nn as _nn
from transformers import TrainerCallback
from peft import get_peft_model_state_dict, set_peft_model_state_dict

RUN_LOG = os.path.join(CFG.WORK_DIR, "run_log.txt")

def log(msg, banner=False):
    """timestamped print (flushed) + copy in run_log.txt"""
    line = f"[{(time.time() - SESSION_T0) / 60:7.1f} min] {msg}"
    if banner:
        line = "\n" + "=" * 110 + "\n" + line + "\n" + "=" * 110
    print(line, flush=True)
    with open(RUN_LOG, "a", encoding="utf-8") as f:
        f.write(line + "\n")

def gpu_mem():
    if not torch.cuda.is_available():
        return "cpu"
    return f"GPU {torch.cuda.memory_allocated() / 1e9:.1f} GB used, peak {torch.cuda.max_memory_allocated() / 1e9:.1f} GB"

SCHEME_KEYS = {"shared": ["ALL"], "class": ["A", "B"], "subclass": ["A1", "A2", "A3", "B"]}

def scheme_key(scheme, grp):
    return "ALL" if scheme == "shared" else (grp if scheme == "subclass" else grp[0])

ACTIVE_SCHEME = "shared"

def ocr_prompt(grp=None):
    """overrides Cell 5 — shared systems: SHARED_PROMPT (+ optional scan-type hint); class / sub-class systems: the
    prompt of the line's class. The same function builds the prompt in training (collator) and decoding."""
    if ACTIVE_SCHEME == "shared" or not grp:
        return CFG.SHARED_PROMPT + (f" Scan type: {grp[0]}." if (CFG.SHARED_CLASS_HINT and grp) else "")
    return CFG.CLASS_PROMPTS[scheme_key(ACTIVE_SCHEME, grp)]

def adapter_name(sysname, key):
    return sysname if key == "ALL" else f"{sysname}__{key}"

def loop_guard(text, max_run=None):
    max_run = max_run or CFG.LOOP_GUARD
    ws, out, run = norm_text(text).split(), [], 0
    for i, w in enumerate(ws):
        run = run + 1 if (i and w == ws[i - 1]) else 1
        if run <= max_run:
            out.append(w)
    return " ".join(out)

# ---- parse / validate SYSTEMS ----
SYS = collections.OrderedDict()
for _s in CFG.SYSTEMS:
    s = {"keys": None, "init": "fresh", "replay": 0.0, "hp": {}, "class_hp": {}, **_s}
    assert s["model"] in CFG.BASE_MODELS, f"{s['name']}: unknown base model {s['model']!r}"
    assert s["scheme"] in SCHEME_KEYS, f"{s['name']}: scheme must be one of {list(SCHEME_KEYS)}"
    s["keys"] = list(s["keys"] or SCHEME_KEYS[s["scheme"]])
    for k in s["keys"]:
        assert k in SCHEME_KEYS[s["scheme"]], f"{s['name']}: key {k!r} is not a {s['scheme']} key"
        assert s["scheme"] == "shared" or k in CFG.CLASS_PROMPTS, f"CLASS_PROMPTS has no prompt for {k!r}"
    if s["init"].startswith("from:"):
        src = s["init"][5:]
        assert src in SYS and SYS[src]["model"] == s["model"] and SYS[src]["scheme"] == "shared", \
            f"{s['name']}: init source {src!r} must be an EARLIER shared system on the same base model"
    SYS[s["name"]] = s
assert CFG.BASELINE_SYSTEM in SYS, "BASELINE_SYSTEM must be one of SYSTEMS"
BASE_ORDER = list(dict.fromkeys(s["model"] for s in SYS.values()))
FOLD_TAG = f"fold{CFG.FOLD}" if CFG.RUN_MODE == "fold0" else "full"

def sys_hp(s, key):
    hp = {k: getattr(CFG, k) for k in ("LEARNING_RATE", "NUM_TRAIN_EPOCHS", "MAX_STEPS", "R", "LORA_ALPHA", "AUG_COPIES",
                                        "PER_DEVICE_TRAIN_BATCH_SIZE", "GRADIENT_ACCUMULATION_STEPS", "WARMUP_RATIO",
                                        "LR_SCHEDULER")}
    hp.update(CFG.MODEL_OVERRIDES.get(s["model"], {}))
    hp.update(s["hp"])
    hp.update(s["class_hp"].get(key, {}))
    return hp

def _in_key(s, key, grp):
    return key == "ALL" or scheme_key(s["scheme"], grp) == key

def covers(s, grp):
    return scheme_key(s["scheme"], grp) in s["keys"]

def rows_for(samples, n_aug, salt):
    rows = [{"image": x["image"], "text": x["text"], "grp": x["grp"], "aug": False, "seed": -1} for x in samples]
    if CFG.USE_AUGMENT:
        for j, x in enumerate(samples):
            for c in range(int(n_aug)):
                rows.append({"image": x["image"], "text": x["text"], "grp": x["grp"], "aug": True,
                             "seed": int((CFG.SEED + 1) * 2000003 + CFG.FOLD * 1000003 + c * 9973 + j + salt * 7777777)
                                     & 0x7fffffff})
    return rows

def adapter_rows(s, key, hp, salt):
    pool = core_samples + pseudo_samples
    own = [x for x in pool if _in_key(s, key, x["grp"])]
    rows = rows_for(own, hp["AUG_COPIES"], salt)
    if s["replay"] > 0 and key != "ALL":
        others = [x for x in pool if not _in_key(s, key, x["grp"])]
        n_rep = min(len(others), int(round(s["replay"] / (1 - s["replay"]) * len(rows))))
        pick = np.random.RandomState(CFG.SEED + 13 + salt).choice(len(others), n_rep, replace=False) if n_rep else []
        rows += [{"image": others[i]["image"], "text": others[i]["text"], "grp": others[i]["grp"], "aug": True,
                  "seed": int(CFG.SEED * 31 + i + salt) & 0x7fffffff} for i in pick]
    return own, rows

# fixed order of the inner slice / val lines so every adapter is validated on the same lines of its class
_INNER_ORDER = [inner_samples[i] for i in np.random.RandomState(CFG.SEED + 7).permutation(len(inner_samples))]
_VAL_ORDER = [val_samples[i] for i in np.random.RandomState(CFG.SEED + 8).permutation(len(val_samples))]

def eval_sets(s, key):
    inner = [x for x in _INNER_ORDER if _in_key(s, key, x["grp"])]
    mon = [x for x in _VAL_ORDER if _in_key(s, key, x["grp"])][:CFG.MONITOR_VAL_N] if CFG.MONITOR_VAL_N else []
    return inner[:CFG.EVAL_GEN_N], inner[:CFG.EVAL_LOSS_N], mon

GREEDY = dict(num_beams=1, repetition_penalty=1.2, no_repeat_ngram_size=0, n_return=1)
EVAL_CURVES, ADAPTER_SUMMARY = [], []

def teacher_forced_loss(model, proc, items):
    tot, n = 0.0, 0
    with torch.no_grad():
        for b in range(0, len(items), CFG.EVAL_BATCH):
            batch = collate_fn([{**x, "aug": False, "seed": -1} for x in items[b:b + CFG.EVAL_BATCH]], proc)
            if batch is None:
                continue
            batch = {k: v.to(model.device) for k, v in batch.items()}
            nt = int((batch["labels"] != -100).sum())
            tot += float(model(**batch).loss) * nt
            n += nt
    return tot / max(1, n)

class StepValidation(TrainerCallback):
    """validates the adapter being trained every CFG.EVAL_EVERY optimizer steps and at the last step; keeps the best
    step's LoRA weights in CPU RAM and restores them at the end (CKPT_METRIC)."""
    def __init__(self, s, key, model, proc):
        self.s, self.key, self.model, self.proc = s, key, model, proc
        self.tag, self.adapter = f"{s['name']}/{key}", adapter_name(s["name"], key)
        self.gen, self.loss_items, self.mon = eval_sets(s, key)
        self.best, self.best_step, self.best_state, self.last_step = None, None, None, -1
        self.train_loss, self.first, self.last = float("nan"), None, None

    def on_log(self, args, state, control, logs=None, **kw):
        if logs and "loss" in logs:
            self.train_loss = float(logs["loss"])

    def validate(self, state):
        if state.global_step == self.last_step:
            return
        self.last_step = state.global_step
        m, t0 = self.model, time.time()
        was_training = m.training
        m.eval()
        keep = (CFG.MAX_NEW_TOKENS, CFG.BATCH_SIZE)
        CFG.MAX_NEW_TOKENS, CFG.BATCH_SIZE = CFG.EVAL_MAX_NEW_TOKENS, CFG.EVAL_BATCH
        res = {}
        try:
            loss = teacher_forced_loss(m, self.proc, self.loss_items) if self.loss_items else float("nan")
            for nm, items in (("inner", self.gen), ("monitor", self.mon)):
                if items:
                    nb = transcribe(m, self.proc, [x["image"] for x in items], GREEDY)
                    res[nm] = official_metric([x["text"] for x in items], [loop_guard(x[0][0]) for x in nb])
        finally:
            CFG.MAX_NEW_TOKENS, CFG.BATCH_SIZE = keep
            if was_training:
                m.train()
        sc = res["inner"]["score"] if "inner" in res else float("nan")
        crit = {"inner_official": sc, "inner_loss": -loss}.get(CFG.CKPT_METRIC)
        flag = ""
        if crit is not None and np.isfinite(crit) and (self.best is None or crit > self.best + 1e-9):
            self.best, self.best_step = crit, state.global_step
            self.best_state = {k: v.detach().to("cpu", copy=True)
                               for k, v in get_peft_model_state_dict(m, adapter_name=self.adapter).items()}
            flag = "  <- best so far"
        row = {"system": self.s["name"], "adapter": self.key, "step": state.global_step, "max_steps": state.max_steps,
               "epoch": state.epoch, "train_loss": self.train_loss, "inner_loss": loss, "inner_score": sc,
               "inner_we": res.get("inner", {}).get("word_edits_per_line", np.nan),
               "inner_ce": res.get("inner", {}).get("char_edits_per_line", np.nan), "n_inner": len(self.gen),
               "monitor_val_score": res.get("monitor", {}).get("score", np.nan), "seconds": time.time() - t0}
        EVAL_CURVES.append(row)
        self.first = self.first or row
        self.last = row
        mon = f" | monitor-val {row['monitor_val_score']:.4f} (n={len(self.mon)}, NOT used)" if self.mon else ""
        log(f"[VALID {self.tag}] step {state.global_step}/{state.max_steps} ep {state.epoch or 0:.2f} | train loss "
            f"{self.train_loss:.4f} | inner loss {loss:.4f} | inner official {sc:.4f} (word/line {row['inner_we']:.3f}, "
            f"char/line {row['inner_ce']:.3f}, n={len(self.gen)}){mon} | {row['seconds']:.0f}s{flag}")

    def on_step_end(self, args, state, control, **kw):
        if CFG.EVAL_EVERY and state.global_step % int(CFG.EVAL_EVERY) == 0:
            self.validate(state)

    def on_train_end(self, args, state, control, **kw):
        self.validate(state)                               # the last step is always validated
        restored = False
        if CFG.CKPT_METRIC != "last" and self.best_state is not None and self.best_step != state.global_step:
            set_peft_model_state_dict(self.model, self.best_state, adapter_name=self.adapter)
            restored = True
            log(f"[CKPT {self.tag}] RESTORED step {self.best_step} ({CFG.CKPT_METRIC} = {self.best:.4f}) instead of the "
                f"last step {state.global_step} (inner official {self.last['inner_score']:.4f})")
        else:
            log(f"[CKPT {self.tag}] keeping the last step {state.global_step}")
        best_row = next((r for r in EVAL_CURVES if r["system"] == self.s["name"] and r["adapter"] == self.key
                         and r["step"] == self.best_step), self.last)
        ADAPTER_SUMMARY.append({"system": self.s["name"], "adapter": self.key, "steps": state.global_step,
                                "best_step": self.best_step, "best_inner_score": best_row["inner_score"],
                                "last_inner_score": self.last["inner_score"], "restored_best": restored})
        self.best_state = None

log(f"TRIAL {TRIAL_ID} | mode {CFG.RUN_MODE} ({FOLD_TAG}) | validation every {CFG.EVAL_EVERY} steps on the inner slice "
    f"({len(inner_samples)} lines) | checkpoint rule {CFG.CKPT_METRIC}", banner=True)
_pool_lines = core_samples + pseudo_samples
for s in SYS.values():
    parts = []
    for k in s["keys"]:
        n_tr = sum(_in_key(s, k, x["grp"]) for x in _pool_lines)
        n_in = sum(_in_key(s, k, x["grp"]) for x in inner_samples)
        n_va = sum(_in_key(s, k, g) for g in VAL_GROUP)
        n_te = sum(_in_key(s, k, g) for g in TEST_GROUP)
        parts.append(f"{k}: train {n_tr} / inner {n_in} / val {n_va} / test {n_te}")
    log(f"SYSTEM {s['name']:14s} base={s['model']:4s} scheme={s['scheme']:8s} init={s['init']:14s} | " + " | ".join(parts))
    if any(sum(_in_key(s, k, x["grp"]) for x in _pool_lines) < 150 for k in s["keys"]):
        log(f"  WARNING {s['name']}: an adapter has fewer than 150 training lines — expect a weak model there")
for k in ("A", "B", "A1", "A2", "A3"):
    if any(s["scheme"] != "shared" and k in s["keys"] for s in SYS.values()):
        log(f"PROMPT[{k}] {CFG.CLASS_PROMPTS[k]}")
log(f"PROMPT[shared] {CFG.SHARED_PROMPT}" + ("  + ' Scan type: A/B.'" if CFG.SHARED_CLASS_HINT else ""))

In [ ]:
# =========================================================
# Cell 13 — for every base model: train its systems' adapters (validated every EVAL_EVERY steps), then decode
#           val / test / unlisted with every system (beam 5, 5-best). One base model resident at a time.
# =========================================================
dtype = torch.bfloat16 if (CFG.USE_BF16 and torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (
        torch.float16 if torch.cuda.is_available() else torch.float32)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NB = {"val": {}, "test": {}, "unlisted": {}}     # split -> system -> per-line N-best ([] = system does not cover the line)
SPLITS = {"val": (VAL_IDS, VAL_PATHS, VAL_GROUP, VAL_GT), "test": (TEST_IDS, TEST_PATHS, TEST_GROUP, None),
          "unlisted": (UNL_IDS, UNL_PATHS, UNL_GROUP, None)}
DONE_KEYS, SYS_MIN = {}, {}
CURRENT_BASE, model, base = None, None, None
TRAIN_SIZE = len(core_samples) + len(pseudo_samples)

def _vision_linear_names(m):
    return [n for n, mod in m.named_modules()
            if isinstance(mod, _nn.Linear) and any(t in n.lower() for t in ("visual", "vision", "merger"))]

def _find_adapter(mname, name):
    roots = ([CFG.ADAPTERS_DIR] if CFG.ADAPTERS_DIR else []) + [CFG.FINAL_DIR] + \
            ([] if CFG.ADAPTERS_DIR else sorted(_glob("/kaggle/input/*")) + sorted(_glob("/kaggle/input/*/*")))
    for r in roots:
        p = os.path.join(r, mname, FOLD_TAG, name)
        if os.path.isfile(os.path.join(p, "adapter_config.json")):
            return p
    for r in roots:                        # exact <model>/<fold>/<adapter> match anywhere below a root (never fuzzy)
        for c in _glob(os.path.join(r, "**", FOLD_TAG, name, "adapter_config.json"), recursive=True):
            parts = os.path.dirname(c).replace("\\", "/").split("/")
            if len(parts) >= 3 and parts[-3] == mname:
                return os.path.dirname(c)
    return None

def attach_adapter(model, base, path, name):
    if model is None:
        return PeftModel.from_pretrained(base, path, adapter_name=name)
    model.load_adapter(path, adapter_name=name)
    return model

def save_nbest(sysname, split, nb):
    ids, _, _, gts = SPLITS[split]
    rows = [{"ID": i, "GroundTruth": gts[k] if gts is not None else None, "pred": nb[k][0][0] if nb[k] else "",
             "logprob": nb[k][0][1] if nb[k] else None, "n_tokens": nb[k][0][2] if nb[k] else None,
             "truncated": nb[k][0][3] if nb[k] else None,
             "nbest": json.dumps([[t, round(sc, 4)] for t, sc, _, _ in nb[k]], ensure_ascii=False)}
            for k, i in enumerate(ids)]
    pd.DataFrame(rows).to_csv(os.path.join(TRIAL_PRED_DIR, f"{TRIAL_ID}__{sysname}__D2_b5nb__{split}.csv"), index=False)

def decode_system(model, proc, s, split):
    global ACTIVE_SCHEME
    ACTIVE_SCHEME = s["scheme"]
    ids, paths, groups, _ = SPLITS[split]
    out = [[] for _ in paths]
    for key in DONE_KEYS.get(s["name"], []):
        idx = [i for i, g in enumerate(groups) if _in_key(s, key, g)]
        if not idx:
            continue
        model.set_adapter(adapter_name(s["name"], key))
        t0 = time.time()
        sub = transcribe(model, proc, [paths[i] for i in idx], CFG.DECODE, desc=f"{split}/{s['name']}/{key}")
        for j, i in enumerate(idx):
            out[i] = [(loop_guard(t), sc, n, tr) for t, sc, n, tr in sub[j]]
        log(f"[DECODE {s['name']}/{key}] {split}: {len(idx)} lines in {(time.time() - t0) / 60:.1f} min "
            f"({sum(sub[j][0][3] for j in range(len(idx)))} truncated)")
    return out

def single_mbr(nb_line):
    return mbr_select([t for t, *_ in nb_line], nbest_posteriors([sc for _, sc, *_ in nb_line], CFG.MBR_TEMPERATURE)) \
        if nb_line else ""

def report_system(sname):
    """val report of one system: top-1 and single-system MBR, per class / group, vs BASELINE_SYSTEM."""
    nb = NB["val"][sname]
    idx = [i for i, x in enumerate(nb) if x]
    top1, mbr = [x[0][0] if x else "" for x in nb], [single_mbr(x) for x in nb]
    base_top = [x[0][0] if x else "" for x in NB["val"].get(CFG.BASELINE_SYSTEM, nb)]
    if len(idx) == len(nb):
        for tag, hyps in (("top1", top1), ("mbr", mbr)):
            report_trial(f"{TRIAL_ID}__{sname}__{tag}", TRIAL_DESCRIPTION, sname, f"D2_b5nb/{tag}", hyps,
                         SYS_MIN.get(sname, 0) * 60, TRAIN_SIZE, sum(x[0][3] for x in nb if x),
                         baseline_hyps=None if (sname == CFG.BASELINE_SYSTEM and tag == "top1") else base_top,
                         extra={"system": sname, "scheme": SYS[sname]["scheme"], "base_model": SYS[sname]["model"]})
    else:                                    # partial system: score only the lines it covers
        g = [VAL_GT[i] for i in idx]
        for tag, hyps in (("top1", top1), ("mbr", mbr)):
            m = official_metric(g, [hyps[i] for i in idx])
            d = paired_delta(g, [base_top[i] for i in idx], [hyps[i] for i in idx])
            log(f"[VAL {sname}/{tag}] covers {len(idx)} val lines: {m['score']:.5f} | vs {CFG.BASELINE_SYSTEM} top-1 on "
                f"those lines {d['delta']:+.5f} CI95 [{d['ci95_lo']:+.5f},{d['ci95_hi']:+.5f}] P(better)={d['p_better']:.3f}")

for btag in BASE_ORDER:
    systems = [s for s in SYS.values() if s["model"] == btag]
    if (time.time() - SESSION_T0) / 3600 > CFG.TIME_BUDGET_H:
        log(f"[TIME] budget {CFG.TIME_BUDGET_H} h reached — skipping base model {btag} ({[s['name'] for s in systems]})")
        continue
    msrc = CFG.BASE_MODELS[btag]
    mname = model_short_name(msrc)
    log(f"BASE MODEL {btag}: {msrc} | systems {[s['name'] for s in systems]} | {gpu_mem()}", banner=True)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    proc = load_vlm_processor(msrc, max_pixels=CFG.MAX_PIXELS, min_pixels=CFG.MIN_PIXELS)
    base = load_vlm_model(msrc, dtype)
    base.to(DEVICE)
    base.config.use_cache = False
    if CFG.DO_TRAIN and any(t in msrc.lower() for t in CFG.GRAD_CKPT_MODELS):
        base.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    base.enable_input_require_grads()
    TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"] + \
              (_vision_linear_names(base) if CFG.LORA_VISION else [])
    ad_root = os.path.join(CFG.FINAL_DIR, mname, FOLD_TAG)
    model, CURRENT_BASE = None, btag
    for s in systems:
        ACTIVE_SCHEME = s["scheme"]
        t_sys = time.time()
        DONE_KEYS[s["name"]] = []
        for kn, key in enumerate(s["keys"]):
            name, hp = adapter_name(s["name"], key), sys_hp(s, key)
            if CFG.DO_TRAIN:
                own, rows = adapter_rows(s, key, hp, salt=list(SYS).index(s["name"]) * 10 + kn + 1)
                if not own:
                    log(f"[TRAIN {s['name']}/{key}] no training lines — adapter skipped (its lines get no vote)")
                    continue
                lcfg = LoraConfig(r=hp["R"], lora_alpha=hp["LORA_ALPHA"], target_modules=TARGETS,
                                  lora_dropout=CFG.LORA_DROPOUT, bias="none", task_type="CAUSAL_LM")
                if model is None:
                    model = get_peft_model(base, lcfg, adapter_name=name)
                else:
                    model.add_adapter(name, lcfg)
                if s["init"].startswith("from:"):
                    src = adapter_name(s["init"][5:], "ALL")
                    set_peft_model_state_dict(model, get_peft_model_state_dict(model, adapter_name=src), adapter_name=name)
                    log(f"[TRAIN {s['name']}/{key}] initialised from adapter {src}")
                model.set_adapter(name)
                n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
                steps = math.ceil(len(rows) / (hp["PER_DEVICE_TRAIN_BATCH_SIZE"] * hp["GRADIENT_ACCUMULATION_STEPS"])) \
                        * hp["NUM_TRAIN_EPOCHS"]
                steps = hp["MAX_STEPS"] if hp["MAX_STEPS"] and hp["MAX_STEPS"] > 0 else steps
                log(f"[TRAIN {s['name']}/{key}] {len(own)} lines -> {len(rows)} rows | lr {hp['LEARNING_RATE']} "
                    f"({hp['LR_SCHEDULER']}) epochs {hp['NUM_TRAIN_EPOCHS']} r {hp['R']} | ~{steps} steps, validation "
                    f"every {CFG.EVAL_EVERY} | trainable {n_trainable / 1e6:.1f}M | {gpu_mem()}")
                cb = StepValidation(s, key, model, proc)
                args = TrainingArguments(
                    output_dir=os.path.join(CFG.OUTPUT_DIR, name),
                    per_device_train_batch_size=hp["PER_DEVICE_TRAIN_BATCH_SIZE"],
                    gradient_accumulation_steps=hp["GRADIENT_ACCUMULATION_STEPS"],
                    learning_rate=hp["LEARNING_RATE"], num_train_epochs=hp["NUM_TRAIN_EPOCHS"],
                    max_steps=int(hp["MAX_STEPS"] or -1), lr_scheduler_type=hp["LR_SCHEDULER"],
                    warmup_ratio=hp["WARMUP_RATIO"], bf16=(dtype == torch.bfloat16), fp16=(dtype == torch.float16),
                    logging_steps=10, eval_strategy="no", save_strategy="no", remove_unused_columns=False,
                    dataloader_num_workers=CFG.DATALOADER_NUM_WORKERS, dataloader_pin_memory=torch.cuda.is_available(),
                    report_to="none", seed=CFG.SEED)
                trainer = Trainer(model=model, args=args, train_dataset=Dataset.from_list(rows),
                                  data_collator=lambda x, _p=proc: collate_fn(x, _p), callbacks=[cb])
                t0 = time.time()
                trainer.train()
                log(f"[TRAIN {s['name']}/{key}] done in {(time.time() - t0) / 60:.1f} min | label-mask failures so far "
                    f"{MASK_FAIL_COUNT} | {gpu_mem()}")
                try:
                    pd.DataFrame(trainer.state.log_history).to_csv(
                        os.path.join(CFG.WORK_DIR, f"log_history__{name}.csv"), index=False)
                except Exception:
                    pass
                os.makedirs(ad_root, exist_ok=True)
                model.save_pretrained(ad_root, selected_adapters=[name])   # -> ad_root/<name>/
                proc.save_pretrained(ad_root)
                del trainer
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                shutil.rmtree(os.path.join(CFG.OUTPUT_DIR, name), ignore_errors=True)
            else:
                p = _find_adapter(mname, name)
                if p is None:
                    log(f"[LOAD {s['name']}/{key}] no saved adapter {name!r} for {mname}/{FOLD_TAG} — skipped")
                    continue
                model = attach_adapter(model, base, p, name)
                log(f"[LOAD {s['name']}/{key}] adapter <- {p}")
            DONE_KEYS[s["name"]].append(key)
        SYS_MIN[s["name"]] = (time.time() - t_sys) / 60
    if model is None:
        log(f"[{btag}] nothing trained or loaded — base model skipped")
        free_cuda("base")
        continue
    model.eval()
    model.config.use_cache = True
    try:
        model.gradient_checkpointing_disable()
    except Exception:
        pass
    if ADAPTER_SUMMARY:
        log("ADAPTER SUMMARY (inner validation):\n" + pd.DataFrame(
            [r for r in ADAPTER_SUMMARY if SYS[r["system"]]["model"] == btag]).round(4).to_string(index=False))
    if CFG.DO_INFER:
        for s in systems:
            if not DONE_KEYS.get(s["name"]):
                continue
            t0 = time.time()
            for split in ("val", "test", "unlisted"):
                if split == "val" and CFG.RUN_MODE != "fold0":
                    continue
                if split == "test" and not CFG.PREDICT_TEST:
                    continue
                if split == "unlisted" and not (CFG.PREDICT_UNLISTED and UNL_IDS):
                    continue
                NB[split][s["name"]] = decode_system(model, proc, s, split)
                save_nbest(s["name"], split, NB[split][s["name"]])
            SYS_MIN[s["name"]] = SYS_MIN.get(s["name"], 0) + (time.time() - t0) / 60
            if CFG.RUN_MODE == "fold0":
                report_system(s["name"])
    pd.DataFrame(EVAL_CURVES).to_csv(os.path.join(CFG.WORK_DIR, "eval_curves.csv"), index=False)
    if btag != BASE_ORDER[-1] or not CFG.RESCORE:        # keep the last base model loaded for the rescoring pass
        free_cuda("model", "base")
        model = base = None
        del proc
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    log(f"[{btag}] finished | {gpu_mem()}")

if ADAPTER_SUMMARY:
    _as = pd.DataFrame(ADAPTER_SUMMARY)
    _as.to_csv(os.path.join(CFG.WORK_DIR, "adapter_summary.csv"), index=False)
    log("ALL ADAPTERS — best vs last inner validation step:\n" + _as.round(4).to_string(index=False))
    _late = _as[_as.best_step < _as.steps]
    if len(_late):
        log(f"{len(_late)} of {len(_as)} adapters peaked BEFORE their last step -> training longer would not help them; "
            f"their best step was restored" if CFG.CKPT_METRIC != "last" else "their last step was kept (CKPT_METRIC='last')")

In [ ]:
# =========================================================
# Cell 14 — CROSS-SYSTEM RESCORING: every system computes the exact log-probability of every pooled candidate
#           (teacher forcing, its own prompt + adapter for the line's class). Measured motivation: the oracle
#           candidate is in only ONE model's 5-best in ~85 % of the lines the pooled MBR gets wrong.
# =========================================================
RESC = {}                                             # split -> list of {"cands": [...], "lp": {system: np.array}}

def pooled_candidates(split):
    n = len(SPLITS[split][0])
    return [list(dict.fromkeys(t for sn in NB[split] for t, *_ in NB[split][sn][i] if t)) or [""] for i in range(n)]

def forced_logprobs(model, proc, path, grp, cands):
    img = load_image(path)
    user = {"role": "user", "content": [{"type": "text", "text": ocr_prompt(grp)}, {"type": "image", "image": img}]}
    tok = proc.tokenizer
    end_id = tok.convert_tokens_to_ids("<|im_end|>")
    prompt_txt = proc.apply_chat_template([user], tokenize=False, add_generation_prompt=True)
    n_prompt = int(proc(text=[prompt_txt], images=[img], return_tensors="pt")["input_ids"].shape[1])
    out = np.full(len(cands), np.nan)
    old = tok.padding_side
    tok.padding_side = "right"
    try:
        for b in range(0, len(cands), CFG.RESCORE_BATCH):
            cs = cands[b:b + CFG.RESCORE_BATCH]
            full = [proc.apply_chat_template([user, {"role": "assistant", "content": [{"type": "text", "text": c}]}],
                                             tokenize=False, add_generation_prompt=False) for c in cs]
            batch = proc(text=full, images=[img] * len(cs), padding=True, return_tensors="pt")
            batch = {k: v.to(model.device) for k, v in batch.items()}
            with torch.inference_mode():
                logits = model(**batch).logits
            ids = batch["input_ids"][:, n_prompt:]
            lp = torch.log_softmax(logits[:, n_prompt - 1:-1].float(), -1).gather(-1, ids.unsqueeze(-1)).squeeze(-1)
            is_end = ids == end_id
            first_end = torch.where(is_end.any(1), is_end.float().argmax(1), torch.full_like(is_end[:, 0], ids.shape[1] - 1,
                                                                                            dtype=torch.long))
            mask = (torch.arange(ids.shape[1], device=ids.device)[None, :] <= first_end[:, None]) & \
                   batch["attention_mask"][:, n_prompt:].bool()
            out[b:b + len(cs)] = (lp * mask).sum(1).cpu().numpy()
            del logits, lp
    finally:
        tok.padding_side = old
    return out

def rescore_system(model, proc, s, split):
    global ACTIVE_SCHEME
    ACTIVE_SCHEME = s["scheme"]
    _, paths, groups, _ = SPLITS[split]
    for key in DONE_KEYS.get(s["name"], []):
        idx = [i for i, g in enumerate(groups) if _in_key(s, key, g) and os.path.exists(paths[i])]
        if not idx:
            continue
        model.set_adapter(adapter_name(s["name"], key))
        t0 = time.time()
        for i in tqdm(idx, desc=f"rescore {split}/{s['name']}/{key}", leave=False):
            try:
                lp = forced_logprobs(model, proc, paths[i], groups[i], RESC[split][i]["cands"])
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                keep, CFG.RESCORE_BATCH = CFG.RESCORE_BATCH, 1
                lp = forced_logprobs(model, proc, paths[i], groups[i], RESC[split][i]["cands"])
                CFG.RESCORE_BATCH = keep
            RESC[split][i]["lp"][s["name"]] = lp
        n_c = np.mean([len(RESC[split][i]["cands"]) for i in idx])
        log(f"[RESCORE {s['name']}/{key}] {split}: {len(idx)} lines x {n_c:.1f} candidates in {(time.time() - t0) / 60:.1f} min")
    # sanity check: the forced log-prob of the system's own top-1 must equal its beam-search log-prob
    diffs = []                       # complete (non-truncated) top-1 only: truncated beams carry no end-token score
    for i, x in enumerate(NB[split][s["name"]]):
        if x and not x[0][3] and s["name"] in RESC[split][i]["lp"] and np.isfinite(x[0][1]):
            j = RESC[split][i]["cands"].index(x[0][0]) if x[0][0] in RESC[split][i]["cands"] else None
            if j is not None:
                diffs.append(abs(RESC[split][i]["lp"][s["name"]][j] - x[0][1]))
        if len(diffs) >= 50:
            break
    if diffs:
        log(f"[RESCORE check {s['name']}] |forced - beam| log-prob of own top-1: median {np.median(diffs):.3f} over "
            f"{len(diffs)} lines" + ("  (OK)" if np.median(diffs) < 0.5 else "  (LARGE: loop-guarded or re-tokenised text; "
                                                                             "rescoring still ranks consistently)"))

if CFG.RESCORE and CFG.DO_INFER and len(NB["test"] or NB["val"]) >= 1:
    splits = [sp for sp in CFG.RESCORE_SPLITS if NB.get(sp)]
    for sp in splits:
        RESC[sp] = [{"cands": c, "lp": {}} for c in pooled_candidates(sp)]
        log(f"[RESCORE] {sp}: {len(RESC[sp])} lines, {np.mean([len(r['cands']) for r in RESC[sp]]):.1f} pooled candidates "
            f"per line from {len(NB[sp])} systems")
    for btag in ([CURRENT_BASE] if model is not None else []) + [b for b in BASE_ORDER if b != CURRENT_BASE or model is None]:
        systems = [s for s in SYS.values() if s["model"] == btag and DONE_KEYS.get(s["name"])]
        if not systems:
            continue
        if (time.time() - SESSION_T0) / 3600 > CFG.TIME_BUDGET_H:
            log(f"[TIME] budget reached — rescoring of {btag} skipped (rescore methods then use the systems scored so far)")
            continue
        msrc = CFG.BASE_MODELS[btag]
        mname = model_short_name(msrc)
        if not (model is not None and btag == CURRENT_BASE):
            free_cuda("model", "base")
            proc = load_vlm_processor(msrc, max_pixels=CFG.MAX_PIXELS, min_pixels=CFG.MIN_PIXELS)
            base = load_vlm_model(msrc, dtype)
            base.to(DEVICE)
            model = None
            for s in systems:
                for key in DONE_KEYS[s["name"]]:
                    p = _find_adapter(mname, adapter_name(s["name"], key))
                    assert p is not None, f"adapter {adapter_name(s['name'], key)} not found for rescoring"
                    model = attach_adapter(model, base, p, adapter_name(s["name"], key))
            model.eval()
            model.config.use_cache = False
            CURRENT_BASE = btag
        log(f"RESCORING with base {btag}: {[s['name'] for s in systems]} | {gpu_mem()}", banner=True)
        for s in systems:
            for sp in splits:
                rescore_system(model, proc, s, sp)
    free_cuda("model", "base")
    model = base = None
    for sp in splits:
        with open(os.path.join(CFG.WORK_DIR, f"rescore__{sp}.jsonl"), "w", encoding="utf-8") as f:
            for i, r in enumerate(RESC[sp]):
                f.write(json.dumps({"ID": SPLITS[sp][0][i], "cands": r["cands"],
                                    "lp": {k: [None if not np.isfinite(v) else round(float(v), 4) for v in a]
                                           for k, a in r["lp"].items()}}, ensure_ascii=False) + "\n")
    log(f"[RESCORE] saved rescore__<split>.jsonl for {splits}")
else:
    free_cuda("model", "base")
    model = base = None
    log("[RESCORE] off (CFG.RESCORE=False) — rescore_* methods are skipped")

In [ ]:
# =========================================================
# Cell 15 — ensembles over ALL systems (each one = a trial on the 819 val lines), champion, submission, pseudo-labels
# =========================================================
try:
    from rapidfuzz.distance import Levenshtein as _RFL
except Exception:
    _RFL = None

def _softmax(x, T=1.0):
    x = np.asarray(x, float)
    m = np.isfinite(x)
    out = np.zeros(len(x))
    if m.any():
        z = x[m] / max(T, 1e-6)
        z = np.exp(z - z.max())
        out[m] = z / z.sum()
    return out

def median_refine(start, cands, w, max_iter=6):
    """extended-hypothesis MBR: greedy single-word edits (taken from other candidates) while the expected official cost
    under the pooled posterior falls. Offline on EXP_002: +0.0003 (P=0.78) over pooled MBR — kept as a challenger."""
    if _RFL is None:
        return start
    ev = [(c, float(x)) for c, x in zip(cands, w) if x > 1e-9]
    risk = lambda h: sum(x * line_cost(c, h) for c, x in ev)
    h, best = start.split(), risk(start)
    for _ in range(max_iter):
        props = set()
        for c, _x in ev:
            cw = c.split()
            for op in _RFL.editops(h, cw):
                nh = (h[:op.src_pos] + [cw[op.dest_pos]] + h[op.src_pos + 1:] if op.tag == "replace" else
                      h[:op.src_pos] + h[op.src_pos + 1:] if op.tag == "delete" else
                      h[:op.src_pos] + [cw[op.dest_pos]] + h[op.src_pos:])
                props.add(" ".join(nh))
        props.discard(" ".join(h))
        if not props:
            break
        cand = min(props, key=risk)
        r = risk(cand)
        if r < best - 1e-12:
            h, best = cand.split(), r
        else:
            break
    return " ".join(h)

def pool_evidence(split, i, weights, systems):
    """own N-best posteriors of every system that covers line i, mixed with the system weights."""
    agg, tot = collections.OrderedDict(), 0.0
    for sn in systems:
        x = NB[split].get(sn, [[]] * (i + 1))[i]
        if not x:
            continue
        p = nbest_posteriors([sc for _, sc, *_ in x], CFG.MBR_TEMPERATURE)
        for (t, *_), q in zip(x, p):
            agg[t] = agg.get(t, 0.0) + weights[sn] * q
        tot += weights[sn]
    if not agg:
        return [""], np.array([1.0])
    return list(agg), np.array(list(agg.values())) / max(tot, 1e-12)

def rescore_evidence(split, i, weights, systems):
    """posterior over the WHOLE pooled candidate list from every system's forced log-probs."""
    r = RESC[split][i]
    w, tot = np.zeros(len(r["cands"])), 0.0
    for sn in systems:
        if sn in r["lp"] and np.isfinite(r["lp"][sn]).any():
            w += weights[sn] * _softmax(r["lp"][sn], CFG.RESCORE_T)
            tot += weights[sn]
    return r["cands"], (w / tot if tot > 0 else np.full(len(r["cands"]), 1.0 / len(r["cands"])))

def run_method(method, split, weights, systems):
    """-> (hyps, risks) for every line of split."""
    n = len(SPLITS[split][0])
    if method.startswith("rescore") and split not in RESC:
        return None, None
    hyps, risks = [], []
    for i in range(n):
        if method.startswith("pool_mbr"):
            cands, w = pool_evidence(split, i, weights, systems)
        else:
            cands, w = rescore_evidence(split, i, weights, systems)
        if method == "rescore_map":
            tot = np.zeros(len(cands)); ok = np.ones(len(cands), bool)
            r = RESC[split][i]
            for sn in systems:
                if sn in r["lp"] and np.isfinite(r["lp"][sn]).any():
                    tot += weights[sn] * np.nan_to_num(r["lp"][sn], nan=-1e4)
            h = cands[int(np.argmax(tot))]
            risk = float(sum(x * line_cost(c, h) for c, x in zip(cands, w)))
        else:
            h, risk = mbr_select(cands, list(w), return_risk=True)
            if method.endswith("_median"):
                h = median_refine(h, cands, w)
                risk = float(sum(x * line_cost(c, h) for c, x in zip(cands, w)))
        hyps.append(h)
        risks.append(risk)
    return hyps, risks

SYSTEMS_DONE = [sn for sn in SYS if sn in NB["test"] or sn in NB["val"]]
CANDIDATES, RISKS, QW, W_FINAL, best, base_top = {}, {}, {}, {}, None, None
if CFG.RUN_MODE == "fold0" and NB["val"]:
    # quality weights fixed by a rule (1 / lost points of each system's single MBR on the val lines it covers)
    QW, EW = {}, {sn: 1.0 for sn in SYSTEMS_DONE}
    for sn in SYSTEMS_DONE:
        idx = [i for i, x in enumerate(NB["val"][sn]) if x]
        sc = official_metric([VAL_GT[i] for i in idx], [single_mbr(NB["val"][sn][i]) for i in idx])["score"]
        QW[sn] = 1.0 / max(1e-4, 1 - sc)
    log("system weights (1/lost points of single-system MBR): " + ", ".join(f"{k}={v:.1f}" for k, v in QW.items()))
    for sn in SYSTEMS_DONE:
        tv = [single_mbr(x) for x in NB["val"][sn]]
        tt = [single_mbr(x) for x in NB["test"][sn]] if sn in NB["test"] else None
        if all(NB["val"][sn]):
            CANDIDATES[f"{sn}__mbr"] = (tv, tt)
            CANDIDATES[f"{sn}__top1"] = ([x[0][0] for x in NB["val"][sn]],
                                         [x[0][0] for x in NB["test"][sn]] if sn in NB["test"] else None)
    base_top = [x[0][0] for x in NB["val"][CFG.BASELINE_SYSTEM]] if CFG.BASELINE_SYSTEM in NB["val"] else None
    RISKS = {}
    for method in CFG.ENSEMBLE_METHODS:
        if method.startswith("rescore") and "val" not in RESC:
            continue
        if method.endswith("_median") and _RFL is None:
            log(f"[ENS] {method} skipped (rapidfuzz missing)"); continue
        wts = EW if method == "pool_mbr_equal" else QW
        t0 = time.time()
        vh, vr = run_method(method, "val", wts, SYSTEMS_DONE)
        th, tr_ = run_method(method, "test", wts, SYSTEMS_DONE) if NB["test"] else (None, None)
        name = f"ENS[{'+'.join(SYSTEMS_DONE)}]__{method}"
        CANDIDATES[name] = (vh, th)
        RISKS[name] = (vr, tr_)
        report_trial(f"{TRIAL_ID}__{method}", TRIAL_DESCRIPTION, "+".join(SYSTEMS_DONE), method, vh, time.time() - t0,
                     TRAIN_SIZE, baseline_hyps=base_top, extra={"system": "ENSEMBLE", "weights": json.dumps(
                         {k: round(v, 2) for k, v in wts.items()})})
        pd.DataFrame({"ID": VAL_IDS, "GroundTruth": VAL_GT, "pred": vh, "risk": vr}).to_csv(
            os.path.join(TRIAL_PRED_DIR, f"{TRIAL_ID}__ENS_{method}__val.csv"), index=False)

    # ---- ranking with per-class / per-group columns and paired bootstrap vs the measured champion recipe ----
    ref_name = next((k for k in CANDIDATES if k.endswith("__pool_mbr_quality")), None)
    rows = []
    for k, (vh, th) in CANDIDATES.items():
        m = official_metric(VAL_GT, vh)
        r = {"candidate": k, "score": m["score"], "word/line": m["word_edits_per_line"], "char/line": m["char_edits_per_line"],
             "has_test": th is not None}
        for c in ("A", "B"):
            ix = [i for i, x in enumerate(VAL_CLASS) if x == c]
            r[c] = official_metric([VAL_GT[i] for i in ix], [vh[i] for i in ix])["score"] if ix else np.nan
        for g in ("A1", "A2", "A3"):
            ix = [i for i, x in enumerate(VAL_GROUP) if x == g]
            r[g] = official_metric([VAL_GT[i] for i in ix], [vh[i] for i in ix])["score"] if ix else np.nan
        if ref_name and k != ref_name:
            d = paired_delta(VAL_GT, CANDIDATES[ref_name][0], vh, n_boot=4000)
            r["vs_pool_mbr_quality"], r["P(better)"] = d["delta"], d["p_better"]
        rows.append(r)
    RANK = pd.DataFrame(rows).sort_values("score", ascending=False)
    RANK.to_csv(os.path.join(CFG.WORK_DIR, "mega_ranking_val.csv"), index=False)
    pd.set_option("display.width", 250)
    log("VALIDATION RANKING (official score, all 819 lines):\n" + RANK.round(5).to_string(index=False), banner=True)

    # ---- how much of the pool's headroom is captured ----
    orc = [min((line_cost(g, t) for sn in SYSTEMS_DONE for t, *_ in NB["val"][sn][i]), default=1.0)
           for i, g in enumerate(VAL_GT)]
    best_single = RANK[~RANK.candidate.str.startswith("ENS[")].score.max()
    best_ens = RANK[RANK.candidate.str.startswith("ENS[")].score.max()
    log(f"POOL ORACLE (best candidate of all systems per line) = {1 - np.mean(orc):.5f} | best single system "
        f"{best_single:.5f} | best ensemble {best_ens:.5f} -> the ensemble captures "
        f"{(best_ens - best_single) / max(1e-9, (1 - np.mean(orc)) - best_single):.0%} of the headroom above the best system")

    # ---- leave-one-system-out: what does each system add? ----
    for method in ("pool_mbr_quality", "rescore_mbr"):
        if method.startswith("rescore") and "val" not in RESC:
            continue
        full = official_metric(VAL_GT, run_method(method, "val", QW, SYSTEMS_DONE)[0])["score"]
        parts = []
        for sn in SYSTEMS_DONE:
            rest = [x for x in SYSTEMS_DONE if x != sn]
            if rest:
                parts.append(f"{sn}: {official_metric(VAL_GT, run_method(method, 'val', QW, rest)[0])['score'] - full:+.5f}")
        log(f"LEAVE-ONE-SYSTEM-OUT for {method} (score change when the system is removed; negative = it helps): "
            + " | ".join(parts))

    best = next((k for k in RANK.candidate if CANDIDATES[k][1] is not None), None)
    if best is not None:
        # "single_mbr:<system>" / "single_top1:<system>" when one system alone wins
        champ_method = best.split("__")[-1] if best.startswith("ENS[") else \
            ("single_" + best.split("__")[-1] + ":" + best.split("__")[0])
        W_FINAL = EW if champ_method == "pool_mbr_equal" else QW
        json.dump({"source": best, "method": champ_method, "weights": W_FINAL,
                   "systems": SYSTEMS_DONE, "val_score": float(RANK.set_index("candidate").at[best, "score"]),
                   "SYSTEMS": CFG.SYSTEMS}, open(os.path.join(CFG.WORK_DIR, "mega_champion.json"), "w"), indent=1, default=str)
        log(f"CHAMPION: {best} (val {RANK.set_index('candidate').at[best, 'score']:.5f}) -> mega_champion.json "
            f"(use it with RUN_MODE='full': FINAL_CHAMPION_JSON=<path>)")
elif CFG.RUN_MODE == "full" and NB["test"]:        # reproduce the Fold-0 champion recipe on test
    _ch = json.load(open(CFG.FINAL_CHAMPION_JSON)) if CFG.FINAL_CHAMPION_JSON and os.path.exists(CFG.FINAL_CHAMPION_JSON) else {}
    method = CFG.FINAL_METHOD or _ch.get("method") or ("rescore_mbr" if RESC.get("test") else "pool_mbr_quality")
    W_FINAL = dict(CFG.FINAL_WEIGHTS or {k: v for k, v in (_ch.get("weights") or {}).items() if k in SYSTEMS_DONE}
                   or {sn: 1.0 for sn in SYSTEMS_DONE})
    for sn in SYSTEMS_DONE:
        W_FINAL.setdefault(sn, float(np.mean(list(W_FINAL.values()))))
    if method.startswith("rescore") and "test" not in RESC:
        log(f"FULL MODE: {method} needs the rescoring pass (RESCORE=True) — falling back to pool_mbr_quality")
        method = "pool_mbr_quality"
    log(f"FULL MODE: method {method} | weights {W_FINAL}")
    if method.startswith("single_"):
        kind, sn = method[len("single_"):].split(":")
        th = [single_mbr(x) if kind == "mbr" else (x[0][0] if x else "") for x in NB["test"][sn]]
        tr_ = [0.0] * len(th)
    else:
        th, tr_ = run_method(method, "test", W_FINAL, SYSTEMS_DONE)
    best = f"ENS[{'+'.join(SYSTEMS_DONE)}]__{method}"
    CANDIDATES[best] = (None, th)
    RISKS = {best: (None, tr_)}

if best is not None and CANDIDATES[best][1] is not None:
    sub = pd.DataFrame({"ID": TEST_IDS, "Target": [norm_text(t) or "the" for t in CANDIDATES[best][1]]})
    sub.to_csv(CFG.SUBMISSION_CSV, index=False)
    log(f"submission.csv <- {best} | rows {len(sub)} | empty replaced {sum(not norm_text(t) for t in CANDIDATES[best][1])}")
    with open(os.path.join(CFG.WORK_DIR, "submission_source.json"), "w") as f:
        json.dump({"trial_session": TRIAL_ID, "source": best}, f)

# ---- pseudo-labels (test + unlisted) with the champion's posterior; risk = expected official cost ----
_pl = []
_meth = best.split("__")[-1] if (best and best.startswith("ENS[")) else "pool_mbr_quality"
_meth = "pool_mbr_quality" if _meth.startswith("single_") else _meth
_w = W_FINAL or {sn: 1.0 for sn in SYSTEMS_DONE}
for sp in ("test", "unlisted"):
    if not NB[sp]:
        continue
    m_ = _meth if (not _meth.startswith("rescore") or sp in RESC) else "pool_mbr_quality"
    hy, rk = run_method(m_, sp, _w, [sn for sn in SYSTEMS_DONE if sn in NB[sp]])
    for i, rid in enumerate(SPLITS[sp][0]):
        _pl.append({"ID": rid, "split": sp, "group": SPLITS[sp][2][i], "Target": hy[i], "risk": rk[i], "method": m_,
                    "n_systems": sum(bool(NB[sp][sn][i]) for sn in NB[sp])})
if _pl:
    _pl = pd.DataFrame(_pl)
    _pl.to_csv(os.path.join(CFG.WORK_DIR, "pseudo_labels.csv"), index=False)
    log(f"pseudo_labels.csv: {len(_pl)} lines | risk quantiles 50/70/90% = "
        f"{np.round(_pl.risk.quantile([.5, .7, .9]).values, 4).tolist()} (calibrate with work/08_build_pseudo_labels.py)")

if os.path.exists(TRIAL_RESULTS) and CFG.RUN_MODE == "fold0":
    res = pd.read_csv(TRIAL_RESULTS)
    sess = res.session == TRIAL_ID
    if sess.any():
        bi = res[sess].score.idxmax()
        res.loc[sess & (res.index != bi) & (res.status == "EVALUATED"), "status"] = "REJECTED"
        res.loc[bi, "status"] = "PROMOTED"
        res.to_csv(TRIAL_RESULTS, index=False)
        log("TRIAL TABLE (this session):\n" + res[sess][["trial", "variant", "score", "score_A", "score_B",
                                                          "word_edits_per_line", "char_edits_per_line", "delta_baseline",
                                                          "p_better_baseline", "status"]].round(5).to_string(index=False))

In [ ]:
# =========================================================
# Cell 16 — forensics of the champion + run summary
# =========================================================
if CFG.RUN_MODE == "fold0" and CANDIDATES and best is not None:
    vh = CANDIDATES[best][0]
    log(f"CHAMPION {best}", banner=True)
    log("per class : " + json.dumps(score_by_group(VAL_GT, vh, VAL_CLASS)))
    log("per group : " + json.dumps(score_by_group(VAL_GT, vh, VAL_GROUP)))
    fx = error_forensics(VAL_GT, vh, top=12)
    for k in ("ops", "word_sub_classes", "word_sub", "char_sub", "char_del", "char_ins"):
        log(f"  {k}: {fx.get(k)}")
    if base_top is not None:
        lb = np.array([line_cost(r, h) for r, h in zip(VAL_GT, base_top)])
        lc = np.array([line_cost(r, h) for r, h in zip(VAL_GT, vh)])
        worst = [i for i in np.argsort(lc - lb)[::-1][:8] if lc[i] > lb[i]]
        if worst:
            log(f"  lines the champion made WORSE than {CFG.BASELINE_SYSTEM} top-1:")
            for i in worst:
                log(f"   {VAL_IDS[i]} [{VAL_GROUP[i]}] GT : {VAL_GT[i]}\n{'':14s}base: {base_top[i]}\n{'':14s}new : {vh[i]}")
if EVAL_CURVES:
    ec = pd.DataFrame(EVAL_CURVES)
    ec.to_csv(os.path.join(CFG.WORK_DIR, "eval_curves.csv"), index=False)
    piv = ec.pivot_table(index="step", columns=["system", "adapter"], values="inner_score")
    log("INNER-VALIDATION CURVES (official score of greedy decoding, by step):\n" + piv.round(4).to_string())
log(f"RUN FINISHED in {(time.time() - SESSION_T0) / 3600:.2f} h | outputs in {CFG.WORK_DIR}: submission.csv, "
    f"trial_results.csv, mega_ranking_val.csv, mega_champion.json, eval_curves.csv, adapter_summary.csv, "
    f"trial_predictions/*__D2_b5nb__<split>.csv, rescore__<split>.jsonl, pseudo_labels.csv, run_log.txt", banner=True)

### After the run
1. **Hand back** `trial_results.csv`, `mega_ranking_val.csv`, `eval_curves.csv`, `adapter_summary.csv`, `run_log.txt`, `trial_predictions/` and `rescore__val.jsonl`. `python work/offline_ensembles.py <dir> --session MEGA_001` re-runs any ensemble offline.
2. **Read the leave-one-system-out lines.** A system whose removal *raises* the score is dead weight: drop it from `SYSTEMS` in the final run.
3. **Final submission:** set `RUN_MODE="full"` and `FINAL_CHAMPION_JSON=<attached Fold-0 output>/mega_champion.json`. Keep the same `SYSTEMS`, minus the dead weight. All systems are retrained on all lines, and the champion method and weights are reproduced on test.
4. **Pseudo-label round:** first run `python work/08_build_pseudo_labels.py` on the `trial_predictions/` folder, then set `PSEUDO_LABEL_CSV=pseudo_labels_final.csv`.

| Knob | When to change it |
|---|---|
| `NUM_TRAIN_EPOCHS=2` + `CKPT_METRIC="inner_official"` | The curves show the score still rising at the last step; the best step is restored anyway |
| `SYSTEMS += 8b_B_cont` | A class-B specialist continued from `8b_shared`. B holds 39 % of the loss |
| `SHARED_CLASS_HINT=True` | One shared model told the era (EXP_003 idea) |
| `RESCORE_SPLITS += ("unlisted",)` | Rescored pseudo-labels for the 687 unlisted images |